<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Notebook 08 — نرمال‌سازی کنترل‌شده فارسی</h2>

<p>
هدف این Notebook ساخت یک فرایند نرمال‌سازی فارسیِ نسخه‌پذیر، قابل‌آزمون و قابل‌ردیابی روی Dataset یکپارچه پذیرفته‌شده در Notebook قبلی است.
</p>

<p>
ورودی اصلی این مرحله Artifact نهایی NB07 یعنی
<code>data/phase2/processed/documents_unified_v1.jsonl</code>
و قرارداد Schema آن یعنی
<code>schemas/phase2/document.schema.json</code>
است.
</p>

<p>
در این Notebook تغییرات مجاز روی متن، از جمله Unicode، نویسه‌های فارسی، فاصله و نیم‌فاصله، نویسه‌های کنترلی، ارقام و علائم، ابتدا به‌صورت Policy مشخص و آزمایش می‌شوند و فقط پس از Validation روی Dataset واقعی اعمال خواهند شد.
</p>

<h3>Scope</h3>

<ul>
<li>Discovery و Revalidation ورودی پذیرفته‌شده NB07</li>
<li>تعریف Policy نرمال‌سازی فارسی بدون تغییر معنای محتوا</li>
<li>حفظ متن ورودی و نسخه نرمال‌شده به‌شکل قابل‌ردیابی</li>
<li>ثبت و اندازه‌گیری Transformationها</li>
<li>آزمون Regression برای موارد فارسی و مالی</li>
<li>اعتبارسنجی مستقل Artifact نهایی</li>
</ul>

<h3>خارج از Scope</h3>

<ul>
<li>Deduplication و Quality Scoring مربوط به NB09</li>
<li>Chunking</li>
<li>Embedding یا Indexing</li>
<li>اصلاح محتوایی، بازنویسی یا حدس Metadata</li>
</ul>

<h3>معیار Done</h3>

<p>
Notebook فقط زمانی پذیرفته می‌شود که ورودی NB07 مستقلاً Revalidate شود، Policy نرمال‌سازی با تست‌های واقعی اثبات شود، Lineage و شناسه‌های اسناد حفظ شوند، حذف یا تغییر غیرعادی متن وجود نداشته باشد و در پایان
<code>Restart Kernel → Run All → Zero Error → Final PASS</code>
و Independent Health Audit موفق باشند.
</p>

<p>
مسیر، نام و Hash خروجی‌های جدید در این مقدمه حدس زده نمی‌شوند و فقط پس از Discovery و تثبیت Contract واقعی تعیین خواهند شد.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Start Gate — کشف مستقل ورودی‌های واقعی NB07</h3>

<p>
پیش از تعریف یا اعمال هر نوع نرمال‌سازی، باید وضعیت واقعی Repository و Artifact پذیرفته‌شده مرحله قبل مستقیماً از روی Disk بررسی شود.
</p>

<p>
این مرحله کاملاً <strong>Read Only</strong> است و هیچ فایل، Schema یا Datasetی را ایجاد، اصلاح یا Overwrite نمی‌کند.
</p>

<p>
در این بررسی، Root واقعی Repository بدون مسیر شخصی Hard-codeشده کشف می‌شود؛ Branch و HEAD فعلی گزارش می‌شوند؛ سپس وجود، اندازه و SHA-256 نوت‌بوک NB07، Dataset یکپارچه و Schema آن از روی فایل‌های واقعی محاسبه می‌شود.
</p>

<p>
همچنین تعداد رکوردهای غیرخالی Dataset مستقیماً از فایل شمارش و با تعداد پذیرفته‌شده NB07 یعنی <code>19456</code> مقایسه می‌شود. اولین رکورد JSON و خود Schema نیز فقط برای اثبات خوانا بودن ورودی Parse می‌شوند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Repository Root با موفقیت کشف شود.</li>
<li>Branch برابر <code>rag-phase2</code> باشد.</li>
<li>Notebook 07، Dataset نهایی و Schema واقعی روی Disk وجود داشته باشند.</li>
<li>هر سه فایل قابل خواندن و Hash شدن باشند.</li>
<li>تعداد رکوردهای غیرخالی Dataset دقیقاً با Gate پذیرفته‌شده NB07 برابر باشد.</li>
<li>اولین رکورد Dataset و فایل Schema JSON معتبر باشند.</li>
</ul>

<p>
اگر هرکدام از این شروط FAIL شود، وارد مرحله بعد نمی‌شویم و همین Markdown و همین Code Cell پس از تشخیص Root Cause اصلاح و Replace خواهند شد.
</p>

</div>

In [1]:
# Code Cell 01

from pathlib import Path
import hashlib
import json
import subprocess


def git_value(project_root, *args):
    return subprocess.check_output(
        ["git", *args],
        cwd=project_root,
        text=True,
    ).strip()


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


current_path = Path.cwd().resolve()

project_root = next(
    (
        candidate
        for candidate in (current_path, *current_path.parents)
        if (candidate / ".git").exists()
        and (candidate / "notebooks" / "phase2").exists()
    ),
    None,
)

assert project_root is not None, "RAG_finance repository root was not found."

branch = git_value(project_root, "rev-parse", "--abbrev-ref", "HEAD")
head = git_value(project_root, "rev-parse", "HEAD")
git_status = git_value(project_root, "status", "--short")

nb07_path = project_root / "notebooks" / "phase2" / "07_unified_schema_and_lineage.ipynb"
dataset_path = project_root / "data" / "phase2" / "processed" / "documents_unified_v1.jsonl"
schema_path = project_root / "schemas" / "phase2" / "document.schema.json"

input_paths = {
    "NB07 notebook": nb07_path,
    "NB07 unified dataset": dataset_path,
    "NB07 document schema": schema_path,
}

print("Repository root:", project_root)
print("Branch:", branch)
print("HEAD:", head)
print("Git status:", "CLEAN" if not git_status else "HAS CHANGES")

if git_status:
    print("\nGit status details:")
    print(git_status)

print("\nInput discovery:")

for label, path in input_paths.items():
    exists = path.is_file()
    print(f"\n{label}:")
    print("  Path:", path.relative_to(project_root))
    print("  Exists:", exists)

    if exists:
        print("  Size bytes:", path.stat().st_size)
        print("  SHA-256:", sha256_file(path))

missing_inputs = [
    label
    for label, path in input_paths.items()
    if not path.is_file()
]

assert branch == "rag-phase2", f"Unexpected branch: {branch}"
assert not missing_inputs, f"Missing required inputs: {missing_inputs}"

record_count = 0
first_record = None

with dataset_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record_count += 1

        if first_record is None:
            first_record = json.loads(line)

with schema_path.open("r", encoding="utf-8") as handle:
    document_schema = json.load(handle)

expected_nb07_record_count = 19456

print("\nDataset discovery:")
print("  Non-empty records:", record_count)
print("  Accepted NB07 records:", expected_nb07_record_count)
print("  First record type:", type(first_record).__name__)
print(
    "  First record keys:",
    sorted(first_record.keys()) if isinstance(first_record, dict) else "NOT A JSON OBJECT",
)

print("\nSchema discovery:")
print("  Schema type:", type(document_schema).__name__)
print("  Schema title:", document_schema.get("title"))
print("  Schema $id:", document_schema.get("$id"))
print(
    "  Required fields:",
    document_schema.get("required", []),
)

assert record_count == expected_nb07_record_count, (
    f"NB07 record-count mismatch: "
    f"expected {expected_nb07_record_count}, found {record_count}"
)
assert isinstance(first_record, dict), "First dataset record is not a JSON object."
assert isinstance(document_schema, dict), "Document schema is not a JSON object."

print("\nNB08 START GATE + INPUT DISCOVERY: PASS")

Repository root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Branch: rag-phase2
HEAD: 6b5c53b04d17c8f14ec66a6b54d3001f67088a2b
Git status: HAS CHANGES

Git status details:
?? notebooks/phase2/08_persian_normalization.ipynb

Input discovery:

NB07 notebook:
  Path: notebooks/phase2/07_unified_schema_and_lineage.ipynb
  Exists: True
  Size bytes: 489994
  SHA-256: 8e1ccd671466f301237ccf51eee53b7cd6576b79b0cbd326ade2eba99f276f39

NB07 unified dataset:
  Path: data/phase2/processed/documents_unified_v1.jsonl
  Exists: True
  Size bytes: 63804489
  SHA-256: 4ad6c3bcb1580d49f692e1d8a0406ee4c08bbf580cbb473e37baa107e89f5f04

NB07 document schema:
  Path: schemas/phase2/document.schema.json
  Exists: True
  Size bytes: 15673
  SHA-256: 3f577c71e8faa3b50c231bd4163cb5425158d0390e5fe5fcf39e00f530baa755

Dataset discovery:
  Non-empty records: 19456
  Accepted NB07 records: 19456
  First record type: dict
  First record keys: ['attribution', 'content_origin', 'document

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Revalidation کامل Dataset ورودی NB07</h3>

<p>
Start Gate وجود و هویت Artifactهای اصلی را تأیید کرد. اکنون قبل از بررسی نویسه‌های فارسی، کل Dataset دوباره از روی Disk خوانده و Contract پایه آن مستقلاً اعتبارسنجی می‌شود.
</p>

<p>
این مرحله همچنان کاملاً <strong>Read Only</strong> است. هیچ متن، Metadata، Schema یا Artifactی تغییر نمی‌کند.
</p>

<p>
برای تمام رکوردها، JSON Schema واقعی پروژه بررسی می‌شود؛ همچنین تکراری نبودن <code>document_id</code>، خالی نبودن متن و توزیع واقعی <code>source_type</code>، <code>schema_version</code> و <code>language</code> اندازه‌گیری می‌شود.
</p>

<p>
هدف این است که اگر در فاصله میان NB07 و NB08 ورودی تغییر یا خراب شده باشد، قبل از هر عملیات نرمال‌سازی متوقف شویم.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام رکوردهای غیرخالی JSON معتبر باشند.</li>
<li>Schema failure برابر صفر باشد.</li>
<li><code>document_id</code> تکراری برابر صفر باشد.</li>
<li>متن خالی برابر صفر باشد.</li>
<li>تعداد رکوردهای Validationشده با تعداد کشف‌شده در Start Gate برابر باشد.</li>
</ul>

<p>
توزیع Source، Language و Schema Version در این مرحله فقط از داده واقعی گزارش می‌شود و هیچ مقدار آن از قبل حدس زده یا Hard-code نمی‌شود.
</p>

</div>

In [2]:
# Code Cell 02

from collections import Counter
from jsonschema import validators

validator_class = validators.validator_for(document_schema)
validator_class.check_schema(document_schema)
document_validator = validator_class(document_schema)

validated_records = 0
json_failures = []
schema_failures = []
document_ids = set()
duplicate_document_ids = []
empty_text_document_ids = []

source_type_counts = Counter()
schema_version_counts = Counter()
language_counts = Counter()

with dataset_path.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            json_failures.append((line_number, str(exc)))
            continue

        validated_records += 1

        errors = sorted(
            document_validator.iter_errors(record),
            key=lambda error: list(error.absolute_path),
        )

        if errors:
            schema_failures.append(
                {
                    "line": line_number,
                    "document_id": record.get("document_id"),
                    "errors": [
                        {
                            "path": ".".join(map(str, error.absolute_path)),
                            "message": error.message,
                        }
                        for error in errors
                    ],
                }
            )

        document_id = record.get("document_id")

        if document_id in document_ids:
            duplicate_document_ids.append(document_id)
        else:
            document_ids.add(document_id)

        text = record.get("text")

        if not isinstance(text, str) or not text.strip():
            empty_text_document_ids.append(document_id)

        source_type_counts[str(record.get("source_type"))] += 1
        schema_version_counts[str(record.get("schema_version"))] += 1
        language_counts[str(record.get("language"))] += 1

print("Records validated:", validated_records)
print("JSON failures:", len(json_failures))
print("Schema failures:", len(schema_failures))
print("Duplicate document_id:", len(duplicate_document_ids))
print("Empty text records:", len(empty_text_document_ids))

print("\nSource type distribution:")
for key, value in sorted(source_type_counts.items()):
    print(f"  {key}: {value}")

print("\nSchema version distribution:")
for key, value in sorted(schema_version_counts.items()):
    print(f"  {key}: {value}")

print("\nLanguage distribution:")
for key, value in sorted(language_counts.items()):
    print(f"  {key}: {value}")

if json_failures:
    print("\nFirst JSON failures:")
    for item in json_failures[:5]:
        print(" ", item)

if schema_failures:
    print("\nFirst schema failures:")
    for item in schema_failures[:3]:
        print(json.dumps(item, ensure_ascii=False, indent=2))

assert validated_records == record_count, (
    f"Validated record count mismatch: "
    f"expected {record_count}, found {validated_records}"
)
assert not json_failures, f"JSON failures found: {len(json_failures)}"
assert not schema_failures, f"Schema failures found: {len(schema_failures)}"
assert not duplicate_document_ids, (
    f"Duplicate document_id values found: {len(duplicate_document_ids)}"
)
assert not empty_text_document_ids, (
    f"Empty text records found: {len(empty_text_document_ids)}"
)

print("\nNB08 INPUT FULL REVALIDATION: PASS")

Records validated: 19456
JSON failures: 0
Schema failures: 0
Duplicate document_id: 0
Empty text records: 0

Source type distribution:
  book: 2701
  transcript: 16226
  website: 529

Schema version distribution:
  common_document_v1: 19456

Language distribution:
  None: 16755
  fa: 2701

NB08 INPUT FULL REVALIDATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery نویسه‌ها و الگوهای واقعی متن فارسی</h3>

<p>
ورودی NB07 از نظر JSON، Schema، شناسه و متن خالی با موفقیت Revalidate شد. اکنون پیش از تعریف Policy نرمال‌سازی، باید مشخص شود در متن واقعی Corpus چه نوع تفاوت‌های Unicode و نویسه‌ای وجود دارد.
</p>

<p>
این مرحله فقط <strong>Discovery</strong> است و هیچ متنی را تغییر نمی‌دهد. هدف، اندازه‌گیری مواردی است که ممکن است در مراحل بعدی نیاز به نرمال‌سازی داشته باشند.
</p>

<p>
موارد بررسی‌شده شامل NFC، BOM، Soft Hyphen، شکل عربی «ی» و «ک»، نیم‌فاصله، ارقام فارسی/عربی/لاتین و چند نویسه کنترلی رایج است. علاوه بر تعداد کل occurrenceها، تعداد Documentهای درگیر نیز گزارش می‌شود تا شدت واقعی هر مورد مشخص باشد.
</p>

<p>
مقدار <code>language=None</code> که در مرحله قبل مشاهده شد در این Cell تغییر نمی‌کند؛ NB08 در این نقطه فقط متن را بررسی می‌کند و از روی محتوای فارسی برای Metadata زبان مقدار حدس نمی‌زند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام <code>19456</code> متن بدون تغییر اسکن شوند.</li>
<li>برای هر الگوی مورد بررسی، occurrence و document coverage از داده واقعی محاسبه شود.</li>
<li>هیچ Artifact یا Datasetی نوشته یا اصلاح نشود.</li>
<li>Discovery بدون Error کامل شود.</li>
</ul>

<p>
وجود نویسه‌های قابل نرمال‌سازی به معنی FAIL نیست؛ نتیجه این Cell مبنای تعریف Policy در مرحله بعد خواهد بود.
</p>

</div>

In [3]:
# Code Cell 03

import unicodedata
from collections import Counter

character_rules = {
    "BOM U+FEFF": "\ufeff",
    "Soft hyphen U+00AD": "\u00ad",
    "Arabic yeh U+064A": "\u064a",
    "Arabic kaf U+0643": "\u0643",
    "ZWNJ U+200C": "\u200c",
    "ZWJ U+200D": "\u200d",
    "LRM U+200E": "\u200e",
    "RLM U+200F": "\u200f",
    "Word joiner U+2060": "\u2060",
    "NBSP U+00A0": "\u00a0",
}

digit_groups = {
    "Persian digits": set("۰۱۲۳۴۵۶۷۸۹"),
    "Arabic-Indic digits": set("٠١٢٣٤٥٦٧٨٩"),
    "ASCII digits": set("0123456789"),
}

occurrence_counts = Counter()
document_counts = Counter()
digit_occurrence_counts = Counter()
digit_document_counts = Counter()

records_scanned = 0
non_nfc_documents = 0
total_characters = 0

with dataset_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        text = record["text"]

        records_scanned += 1
        total_characters += len(text)

        if not unicodedata.is_normalized("NFC", text):
            non_nfc_documents += 1

        for label, character in character_rules.items():
            count = text.count(character)

            if count:
                occurrence_counts[label] += count
                document_counts[label] += 1

        text_characters = set(text)

        for label, digit_set in digit_groups.items():
            count = sum(text.count(character) for character in digit_set)

            if count:
                digit_occurrence_counts[label] += count

            if text_characters & digit_set:
                digit_document_counts[label] += 1

print("Records scanned:", records_scanned)
print("Total text characters:", total_characters)
print("Non-NFC documents:", non_nfc_documents)

print("\nCharacter discovery:")
for label in character_rules:
    print(
        f"  {label}: "
        f"occurrences={occurrence_counts[label]}, "
        f"documents={document_counts[label]}"
    )

print("\nDigit discovery:")
for label in digit_groups:
    print(
        f"  {label}: "
        f"occurrences={digit_occurrence_counts[label]}, "
        f"documents={digit_document_counts[label]}"
    )

assert records_scanned == record_count, (
    f"Scanned record count mismatch: expected {record_count}, "
    f"found {records_scanned}"
)

print("\nNB08 TEXT CHARACTER DISCOVERY: PASS")

Records scanned: 19456
Total text characters: 3125877
Non-NFC documents: 33

Character discovery:
  BOM U+FEFF: occurrences=205, documents=107
  Soft hyphen U+00AD: occurrences=0, documents=0
  Arabic yeh U+064A: occurrences=137, documents=41
  Arabic kaf U+0643: occurrences=45, documents=17
  ZWNJ U+200C: occurrences=59256, documents=10756
  ZWJ U+200D: occurrences=1, documents=1
  LRM U+200E: occurrences=72, documents=55
  RLM U+200F: occurrences=13, documents=6
  Word joiner U+2060: occurrences=9, documents=2
  NBSP U+00A0: occurrences=124, documents=73

Digit discovery:
  Persian digits: occurrences=1594, documents=308
  Arabic-Indic digits: occurrences=6, documents=2
  ASCII digits: occurrences=6883, documents=1444

NB08 TEXT CHARACTER DISCOVERY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بررسی Context واقعی نویسه‌های حساس قبل از تعریف Policy</h3>

<p>
Discovery قبلی نشان داد چند نوع نویسه و نمایش عدد در Corpus وجود دارد، اما صرف مشاهده تعداد آن‌ها برای تصمیم نرمال‌سازی کافی نیست.
</p>

<p>
در این مرحله نمونه‌های واقعی اطراف نویسه‌های حساس مستقیماً از Dataset استخراج می‌شوند تا مشخص شود هر مورد در چه Contextی ظاهر شده است. این بررسی به‌ویژه برای <code>ZWJ</code>، <code>LRM</code>، <code>RLM</code>، <code>Word Joiner</code>، <code>NBSP</code> و انواع ارقام لازم است؛ زیرا حذف یا تبدیل کورکورانه آن‌ها می‌تواند متن، عدد، URL یا Locator معنایی را تغییر دهد.
</p>

<p>
این Cell همچنان فقط <strong>Read Only</strong> است و هیچ Transformationی اعمال نمی‌کند.
</p>

<h4>خروجی مورد انتظار</h4>

<ul>
<li>چند نمونه محدود برای هر نویسه‌ای که واقعاً در Corpus وجود دارد.</li>
<li><code>document_id</code> و <code>source_type</code> هر نمونه.</li>
<li>Context کوتاه و قابل‌خواندن اطراف نویسه.</li>
<li>نمونه‌هایی از ارقام فارسی، عربی و ASCII.</li>
</ul>

<h4>معیار PASS</h4>

<p>
نمونه‌ها بدون تغییر Dataset استخراج شوند و برای تمام گروه‌های موجود، Context واقعی کافی برای تصمیم Policy مرحله بعد فراهم شود.</p>

</div>

In [4]:
# Code Cell 04

context_targets = {
    "BOM U+FEFF": "\ufeff",
    "Arabic yeh U+064A": "\u064a",
    "Arabic kaf U+0643": "\u0643",
    "ZWJ U+200D": "\u200d",
    "LRM U+200E": "\u200e",
    "RLM U+200F": "\u200f",
    "Word joiner U+2060": "\u2060",
    "NBSP U+00A0": "\u00a0",
}

digit_targets = {
    "Persian digits": set("۰۱۲۳۴۵۶۷۸۹"),
    "Arabic-Indic digits": set("٠١٢٣٤٥٦٧٨٩"),
    "ASCII digits": set("0123456789"),
}

max_samples_per_group = 5
context_radius = 35

character_samples = {label: [] for label in context_targets}
digit_samples = {label: [] for label in digit_targets}
non_nfc_samples = []

with dataset_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        text = record["text"]
        document_id = record["document_id"]
        source_type = record["source_type"]

        if (
            not unicodedata.is_normalized("NFC", text)
            and len(non_nfc_samples) < max_samples_per_group
        ):
            normalized = unicodedata.normalize("NFC", text)

            first_difference = next(
                (
                    index
                    for index, (before, after) in enumerate(zip(text, normalized))
                    if before != after
                ),
                0,
            )

            start = max(0, first_difference - context_radius)
            end = min(len(text), first_difference + context_radius + 1)

            non_nfc_samples.append(
                {
                    "document_id": document_id,
                    "source_type": source_type,
                    "context": repr(text[start:end]),
                }
            )

        for label, target in context_targets.items():
            if len(character_samples[label]) >= max_samples_per_group:
                continue

            search_start = 0

            while len(character_samples[label]) < max_samples_per_group:
                index = text.find(target, search_start)

                if index == -1:
                    break

                start = max(0, index - context_radius)
                end = min(len(text), index + context_radius + 1)

                character_samples[label].append(
                    {
                        "document_id": document_id,
                        "source_type": source_type,
                        "context": repr(text[start:end]),
                    }
                )

                search_start = index + 1

        for label, digit_set in digit_targets.items():
            if len(digit_samples[label]) >= max_samples_per_group:
                continue

            for index, character in enumerate(text):
                if character not in digit_set:
                    continue

                start = max(0, index - context_radius)
                end = min(len(text), index + context_radius + 1)

                digit_samples[label].append(
                    {
                        "document_id": document_id,
                        "source_type": source_type,
                        "context": repr(text[start:end]),
                    }
                )

                if len(digit_samples[label]) >= max_samples_per_group:
                    break

print("Non-NFC samples:")
for sample in non_nfc_samples:
    print(
        f"  [{sample['source_type']}] {sample['document_id']} -> "
        f"{sample['context']}"
    )

print("\nCharacter context samples:")
for label, samples in character_samples.items():
    print(f"\n{label} | samples={len(samples)}")
    for sample in samples:
        print(
            f"  [{sample['source_type']}] {sample['document_id']} -> "
            f"{sample['context']}"
        )

print("\nDigit context samples:")
for label, samples in digit_samples.items():
    print(f"\n{label} | samples={len(samples)}")
    for sample in samples:
        print(
            f"  [{sample['source_type']}] {sample['document_id']} -> "
            f"{sample['context']}"
        )

print("\nNB08 CHARACTER CONTEXT INSPECTION: PASS")

Non-NFC samples:
  [book] book:adult-financial-literacy-course:fa-source:p000084 -> 'درس ۲۵: مؤلفه\u200cهای مالی در تصمیم\u200cگیری مالی57'
  [book] book:adult-financial-literacy-course:fa-source:p000089 -> 'درس ۲۶: مؤلفه\u200cهای غیرمالی در تصمیم\u200cگیری مالی'
  [book] book:adult-financial-literacy-course:fa-source:p000118 -> 'عضو هیأت علمی آکادمی هوش مالی'
  [book] book:adult-financial-literacy-course:fa-source:p000123 -> 'بعد معنوی\u200cروحی: مأموریت فردی، چشم\u200cانداز، اهداف کلان،'
  [book] book:adult-financial-literacy-course:fa-source:p000495 -> ' از این شیوه\u200cها برای دریافت پول و تأمین مالی کارها بهره برد.'

Character context samples:

BOM U+FEFF | samples=5
  [book] book:adult-financial-literacy-course:fa-source:p000536 -> 'حوزه\ufeffهای بهبود در توسعه فردی مشخص می\u200cشود'
  [book] book:adult-financial-literacy-course:fa-source:p000538 -> 'راه\ufeffهای ورود به کار باز شده و ارتباطات '
  [book] book:adult-financial-literacy-course:fa-source:p001149 -> 'بیمه\ufeffگ

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Policy نهایی Context-aware برای نویسه‌های کنترلی</h3>

<p>
بازبینی Corpus نشان داد نویسه‌های کنترلی Unicode را نمی‌توان به‌صورت سراسری حذف کرد؛ زیرا بعضی از آن‌ها در داده واقعی نقش مرز واژه و نیم‌فاصله را گرفته‌اند.
</p>

<p>
بنابراین Policy این نسخه براساس Context سمت چپ و راست هر نویسه عمل می‌کند و در الگوهای ناشناخته به‌صورت محافظه‌کارانه نویسه را حفظ می‌کند.
</p>

<h4>قواعد اصلی</h4>

<ul>
<li><code>FEFF</code> بین دو حرف به <code>ZWNJ</code> تبدیل می‌شود.</li>
<li><code>LRM</code> بین دو حرف به <code>ZWNJ</code> تبدیل می‌شود.</li>
<li>اگر <code>LRM/RLM/ZWJ/Word Joiner</code> کنار ZWNJ موجود باشند، Control حذف و ZWNJ موجود حفظ می‌شود.</li>
<li>Controlهای Formatting کنار Space، Punctuation یا Mark فقط در الگوهای مشاهده‌شده حذف می‌شوند.</li>
<li>الگوی ناشناخته به‌صورت خودکار حذف نمی‌شود.</li>
<li><code>ي</code> و <code>ك</code> عربی همچنان حفظ می‌شوند.</li>
<li><code>NBSP</code> با Rule زمینه‌آگاه مدیریت می‌شود.</li>
<li>ارقام Arabic-Indic به ارقام فارسی تبدیل می‌شوند؛ ارقام ASCII حفظ می‌شوند.</li>
</ul>

<p>
این Policy نتیجه مستقیم Discovery و Failureهای واقعی همین Notebook است و جایگزین Policy حذف سراسری Control Characterها می‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>برای تمام Control Characterهای مشاهده‌شده Rule صریح وجود داشته باشد.</li>
<li>Default Action برای Context ناشناخته <code>preserve</code> باشد.</li>
<li>تبدیل <code>ي/ك</code> سراسری ممنوع باقی بماند.</li>
<li>هیچ Dataset یا Artifactی در این مرحله نوشته نشود.</li>
</ul>

</div>

In [5]:
# Code Cell 05

normalization_policy = {
    "policy_version": "persian_normalization_v1",
    "policy_revision": "contextual_controls_r2",
    "unicode_form": "NFC",
    "text_field_scope": ["text"],

    "transformations": {
        "soft_hyphen": {
            "character": "\u00ad",
            "action": "remove",
        },

        "control_characters": {
            "\ufeff": {
                "label": "BOM U+FEFF",
                "rules": {
                    ("LETTER", "LETTER"): "replace_with_zwnj",
                },
                "default_action": "preserve",
            },

            "\u200d": {
                "label": "ZWJ U+200D",
                "rules": {
                    ("ZWNJ", "LETTER"): "remove",
                },
                "default_action": "preserve",
            },

            "\u200e": {
                "label": "LRM U+200E",
                "rules": {
                    ("LETTER", "LETTER"): "replace_with_zwnj",
                    ("LETTER", "ZWNJ"): "remove",
                    ("ZWNJ", "LETTER"): "remove",
                    ("LETTER", "SPACE"): "remove",
                    ("SPACE", "LETTER"): "remove",
                    ("LETTER", "PUNCT"): "remove",
                },
                "default_action": "preserve",
            },

            "\u200f": {
                "label": "RLM U+200F",
                "rules": {
                    ("LETTER", "ZWNJ"): "remove",
                    ("ZWNJ", "LETTER"): "remove",
                    ("MARK", "SPACE"): "remove",
                },
                "default_action": "preserve",
            },

            "\u2060": {
                "label": "Word joiner U+2060",
                "rules": {
                    ("ZWNJ", "LETTER"): "remove",
                    ("LETTER", "ZWNJ"): "remove",
                },
                "default_action": "preserve",
            },
        },

        "contextual_rules": {
            "nbsp_boundary": {
                "source": "\u00a0",
                "target": " ",
                "collapse_spacing_run_containing_nbsp": True,
            },
        },

        "digit_mapping": {
            "٠": "۰",
            "١": "۱",
            "٢": "۲",
            "٣": "۳",
            "٤": "۴",
            "٥": "۵",
            "٦": "۶",
            "٧": "۷",
            "٨": "۸",
            "٩": "۹",
        },
    },

    "preserve": {
        "zwnj_u200c": True,
        "arabic_yeh_u064a": True,
        "arabic_kaf_u0643": True,
        "persian_digits": True,
        "ascii_digits": True,
        "newlines": True,
        "punctuation": True,
        "unknown_control_contexts": True,
    },

    "forbidden_operations": [
        "remove_control_characters_globally",
        "remove_zwnj_globally",
        "convert_arabic_yeh_globally",
        "convert_arabic_kaf_globally",
        "convert_ascii_digits_globally",
        "collapse_whitespace_globally",
        "rewrite_words",
        "infer_or_modify_language_metadata",
        "modify_urls",
        "modify_document_ids",
        "modify_source_ids",
        "modify_lineage",
        "modify_attribution",
        "modify_rights",
    ],
}

control_rules = normalization_policy["transformations"]["control_characters"]

print("Policy version:", normalization_policy["policy_version"])
print("Policy revision:", normalization_policy["policy_revision"])
print("Unicode form:", normalization_policy["unicode_form"])
print("Control characters covered:", len(control_rules))
print(
    "Unknown control contexts:",
    "PRESERVE"
    if normalization_policy["preserve"]["unknown_control_contexts"]
    else "NOT PRESERVED",
)

print("\nControl-character policy:")

for character, config in control_rules.items():
    print(
        f"\n{config['label']} | {repr(character)}"
    )

    for pattern, action in config["rules"].items():
        print(
            f"  {pattern[0]} | CONTROL | {pattern[1]}"
            f" -> {action}"
        )

    print(
        "  DEFAULT ->",
        config["default_action"],
    )

assert normalization_policy["unicode_form"] == "NFC"

assert normalization_policy["preserve"]["zwnj_u200c"] is True
assert normalization_policy["preserve"]["arabic_yeh_u064a"] is True
assert normalization_policy["preserve"]["arabic_kaf_u0643"] is True
assert normalization_policy["preserve"]["ascii_digits"] is True
assert normalization_policy["preserve"]["unknown_control_contexts"] is True

assert control_rules["\ufeff"]["rules"][
    ("LETTER", "LETTER")
] == "replace_with_zwnj"

assert control_rules["\u200e"]["rules"][
    ("LETTER", "LETTER")
] == "replace_with_zwnj"

assert all(
    config["default_action"] == "preserve"
    for config in control_rules.values()
)

assert (
    "remove_control_characters_globally"
    in normalization_policy["forbidden_operations"]
)

print("\nNB08 CONTEXT-AWARE NORMALIZATION POLICY: PASS")

Policy version: persian_normalization_v1
Policy revision: contextual_controls_r2
Unicode form: NFC
Control characters covered: 5
Unknown control contexts: PRESERVE

Control-character policy:

BOM U+FEFF | '\ufeff'
  LETTER | CONTROL | LETTER -> replace_with_zwnj
  DEFAULT -> preserve

ZWJ U+200D | '\u200d'
  ZWNJ | CONTROL | LETTER -> remove
  DEFAULT -> preserve

LRM U+200E | '\u200e'
  LETTER | CONTROL | LETTER -> replace_with_zwnj
  LETTER | CONTROL | ZWNJ -> remove
  ZWNJ | CONTROL | LETTER -> remove
  LETTER | CONTROL | SPACE -> remove
  SPACE | CONTROL | LETTER -> remove
  LETTER | CONTROL | PUNCT -> remove
  DEFAULT -> preserve

RLM U+200F | '\u200f'
  LETTER | CONTROL | ZWNJ -> remove
  ZWNJ | CONTROL | LETTER -> remove
  MARK | CONTROL | SPACE -> remove
  DEFAULT -> preserve

Word joiner U+2060 | '\u2060'
  ZWNJ | CONTROL | LETTER -> remove
  LETTER | CONTROL | ZWNJ -> remove
  DEFAULT -> preserve

NB08 CONTEXT-AWARE NORMALIZATION POLICY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>پیاده‌سازی Normalizer زمینه‌آگاه و Regression Test</h3>

<p>
Policy جدید برای هر Control Character براساس نوع نویسه سمت چپ و راست تصمیم می‌گیرد. بنابراین دیگر حذف سراسری انجام نمی‌شود.
</p>

<p>
برای مثال:</p>

<ul>
<li><code>حوزه﻿های</code> به <code>حوزه‌های</code> تبدیل می‌شود.</li>
<li><code>درس‎های</code> به <code>درس‌های</code> تبدیل می‌شود.</li>
<li>اگر ZWNJ از قبل وجود داشته باشد، Control اضافی حذف و همان ZWNJ حفظ می‌شود.</li>
<li>اگر Context در Policy شناخته‌شده نباشد، Control Character دست‌نخورده باقی می‌ماند.</li>
</ul>

<p>
همچنین Ruleهای قبلی برای NFC، NBSP، Soft Hyphen و ارقام Arabic-Indic حفظ می‌شوند و تبدیل سراسری <code>ي/ك</code> همچنان ممنوع است.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام Unit و Regression Testها PASS شوند.</li>
<li>نمونه‌های واقعی <code>FEFF</code> و <code>LRM</code> مرز واژه صحیح تولید کنند.</li>
<li>ZWNJ موجود حفظ شود و ZWNJ اضافه تولید نشود.</li>
<li>Context ناشناخته Preserve شود.</li>
<li>متن عربی و <code>ي/ك</code> تغییر نکنند.</li>
<li>Normalizer Idempotent باشد.</li>
</ul>

<p>
در صورت FAIL، همین Markdown و همین <code>Code Cell 06</code> Replace می‌شوند و وارد Corpus نمی‌شویم.
</p>

</div>

In [6]:
# Code Cell 06

import re


def classify_neighbor(character):
    if character is None:
        return "BOUNDARY"

    if character == "\u200c":
        return "ZWNJ"

    if character.isspace():
        return "SPACE"

    category = unicodedata.category(character)

    if category.startswith("L"):
        return "LETTER"

    if category.startswith("N"):
        return "DIGIT"

    if category.startswith("P"):
        return "PUNCT"

    if category.startswith("M"):
        return "MARK"

    return f"OTHER:{category}"


def apply_contextual_control_rules(text, policy):
    control_rules = policy["transformations"]["control_characters"]
    output = []

    for index, character in enumerate(text):
        config = control_rules.get(character)

        if config is None:
            output.append(character)
            continue

        left = text[index - 1] if index > 0 else None
        right = text[index + 1] if index + 1 < len(text) else None

        pattern = (
            classify_neighbor(left),
            classify_neighbor(right),
        )

        action = config["rules"].get(
            pattern,
            config["default_action"],
        )

        if action == "remove":
            continue

        if action == "replace_with_zwnj":
            output.append("\u200c")
            continue

        if action == "preserve":
            output.append(character)
            continue

        raise ValueError(
            f"Unknown control action {action!r} "
            f"for {config['label']} and pattern {pattern}"
        )

    return "".join(output)


def normalize_persian_text(text, policy=normalization_policy):
    if not isinstance(text, str):
        raise TypeError(
            f"text must be str, got {type(text).__name__}"
        )

    normalized = unicodedata.normalize(
        policy["unicode_form"],
        text,
    )

    soft_hyphen = policy["transformations"]["soft_hyphen"]

    if soft_hyphen["action"] == "remove":
        normalized = normalized.replace(
            soft_hyphen["character"],
            "",
        )
    else:
        raise ValueError(
            f"Unsupported soft-hyphen action: "
            f"{soft_hyphen['action']}"
        )

    normalized = apply_contextual_control_rules(
        normalized,
        policy,
    )

    nbsp_rule = (
        policy["transformations"]
        ["contextual_rules"]
        ["nbsp_boundary"]
    )

    nbsp = nbsp_rule["source"]

    if nbsp_rule["collapse_spacing_run_containing_nbsp"]:
        normalized = re.sub(
            rf"[ {re.escape(nbsp)}]*"
            rf"{re.escape(nbsp)}"
            rf"[ {re.escape(nbsp)}]*",
            nbsp_rule["target"],
            normalized,
        )
    else:
        normalized = normalized.replace(
            nbsp,
            nbsp_rule["target"],
        )

    for source, target in (
        policy["transformations"]["digit_mapping"].items()
    ):
        normalized = normalized.replace(
            source,
            target,
        )

    return normalized


test_cases = [
    {
        "name": "NFC composition",
        "input": "مؤلف",
        "expected": "مؤلف",
    },
    {
        "name": "FEFF between letters becomes ZWNJ",
        "input": "حوزه\ufeffهای",
        "expected": "حوزه\u200cهای",
    },
    {
        "name": "FEFF compound word becomes ZWNJ",
        "input": "بیمه\ufeffگر",
        "expected": "بیمه\u200cگر",
    },
    {
        "name": "LRM between letters becomes ZWNJ",
        "input": "درس\u200eهای",
        "expected": "درس\u200cهای",
    },
    {
        "name": "LRM verb boundary becomes ZWNJ",
        "input": "می\u200eکند",
        "expected": "می\u200cکند",
    },
    {
        "name": "LRM before existing ZWNJ removed",
        "input": "پس\u200e\u200cانداز",
        "expected": "پس\u200cانداز",
    },
    {
        "name": "LRM after existing ZWNJ removed",
        "input": "سازمان\u200c\u200eهای",
        "expected": "سازمان\u200cهای",
    },
    {
        "name": "LRM before space removed",
        "input": "آموزشی\u200e تولید",
        "expected": "آموزشی تولید",
    },
    {
        "name": "LRM after space removed",
        "input": "هدف. \u200eرسیدن",
        "expected": "هدف. رسیدن",
    },
    {
        "name": "LRM before punctuation removed",
        "input": "بدهی\u200e، کاهش",
        "expected": "بدهی، کاهش",
    },
    {
        "name": "RLM before existing ZWNJ removed",
        "input": "مالی\u200f\u200cشان",
        "expected": "مالی\u200cشان",
    },
    {
        "name": "RLM after existing ZWNJ removed",
        "input": "آورده\u200c\u200fاید",
        "expected": "آورده\u200cاید",
    },
    {
        "name": "Word joiner before ZWNJ removed",
        "input": "می\u2060\u200cگذارد",
        "expected": "می\u200cگذارد",
    },
    {
        "name": "Word joiner after ZWNJ removed",
        "input": "زندگی\u200c\u2060مان",
        "expected": "زندگی\u200cمان",
    },
    {
        "name": "ZWJ after ZWNJ removed",
        "input": "بخت\u200c\u200dآزمایی",
        "expected": "بخت\u200cآزمایی",
    },
    {
        "name": "Unknown LRM context preserved",
        "input": "1\u200eالف",
        "expected": "1\u200eالف",
    },
    {
        "name": "Soft hyphen removed",
        "input": "سرمایه\u00adگذاری",
        "expected": "سرمایهگذاری",
    },
    {
        "name": "NBSP spacing run collapsed",
        "input": "گردون\xa0 \xa0 \xa0 از او",
        "expected": "گردون از او",
    },
    {
        "name": "NBSP around slash normalized",
        "input": "مختلف\xa0 \xa0/\xa0 \xa0 آن یکی",
        "expected": "مختلف / آن یکی",
    },
    {
        "name": "Ordinary double spaces preserved",
        "input": "مالی  و اقتصادی",
        "expected": "مالی  و اقتصادی",
    },
    {
        "name": "Arabic yeh and kaf preserved",
        "input": "كتاب مالي",
        "expected": "كتاب مالي",
    },
    {
        "name": "Arabic-Indic digits to Persian",
        "input": "ص٣٧٧ و ص٥٤٥",
        "expected": "ص۳۷۷ و ص۵۴۵",
    },
    {
        "name": "ASCII digits preserved",
        "input": "2011 و 2016",
        "expected": "2011 و 2016",
    },
    {
        "name": "Persian digits preserved",
        "input": "۹۰ درصد و ۵۸ درصد",
        "expected": "۹۰ درصد و ۵۸ درصد",
    },
    {
        "name": "Newline preserved",
        "input": "خط اول\nخط دوم",
        "expected": "خط اول\nخط دوم",
    },
    {
        "name": "Existing ZWNJ preserved",
        "input": "می\u200cتواند",
        "expected": "می\u200cتواند",
    },
]

test_failures = []

for case in test_cases:
    actual = normalize_persian_text(
        case["input"]
    )

    if actual != case["expected"]:
        test_failures.append(
            {
                "name": case["name"],
                "input": repr(case["input"]),
                "expected": repr(case["expected"]),
                "actual": repr(actual),
            }
        )

idempotence_failures = []

for case in test_cases:
    once = normalize_persian_text(
        case["input"]
    )
    twice = normalize_persian_text(
        once
    )

    if once != twice:
        idempotence_failures.append(
            {
                "name": case["name"],
                "once": repr(once),
                "twice": repr(twice),
            }
        )

print(
    "Unit and regression tests:",
    len(test_cases),
)
print(
    "Test failures:",
    len(test_failures),
)
print(
    "Idempotence tests:",
    len(test_cases),
)
print(
    "Idempotence failures:",
    len(idempotence_failures),
)

if test_failures:
    print("\nTest failure details:")
    for failure in test_failures:
        print(
            json.dumps(
                failure,
                ensure_ascii=False,
                indent=2,
            )
        )

if idempotence_failures:
    print("\nIdempotence failure details:")
    for failure in idempotence_failures:
        print(
            json.dumps(
                failure,
                ensure_ascii=False,
                indent=2,
            )
        )

assert not test_failures, (
    f"Normalization test failures: "
    f"{len(test_failures)}"
)

assert not idempotence_failures, (
    f"Normalization idempotence failures: "
    f"{len(idempotence_failures)}"
)

print(
    "\nNB08 CONTEXT-AWARE NORMALIZER "
    "REGRESSION TESTS: PASS"
)

Unit and regression tests: 26
Test failures: 0
Idempotence tests: 26
Idempotence failures: 0

NB08 CONTEXT-AWARE NORMALIZER REGRESSION TESTS: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Dry Run کامل Normalizer زمینه‌آگاه روی Corpus</h3>

<p>
Normalizer زمینه‌آگاه تمام Unit و Regression Testها را PASS کرده است. اکنون همین تابع روی کل Dataset اجرا می‌شود، اما همچنان فقط در حافظه و بدون ساخت یا تغییر Artifact.
</p>

<p>
این مرحله بررسی می‌کند که Ruleهای جدید روی داده واقعی نیز رفتار ایمن داشته باشند و Control Character ناشناخته‌ای بدون تصمیم صریح تغییر نکند.
</p>

<h4>Validation</h4>

<ul>
<li>تمام <code>19456</code> رکورد پردازش شوند.</li>
<li>هیچ متن غیرخالی، خالی نشود.</li>
<li>تمام خروجی‌ها NFC و Idempotent باشند.</li>
<li><code>ZWNJ</code> موجود از بین نرود.</li>
<li>ارقام ASCII و <code>ي/ك</code> عربی حفظ شوند.</li>
<li><code>NBSP</code> و Arabic-Indic digits باقی نمانند.</li>
<li>Soft Hyphen باقی نماند.</li>
<li>هیچ Control Character با Context ناشناخته به‌طور ناخواسته تغییر نکند.</li>
<li>تعداد فاصله‌های دوبل افزایش پیدا نکند.</li>
</ul>

<h4>معیار PASS</h4>

<p>
تمام Gateها باید PASS شوند. وجود Control Character باقی‌مانده در Context ناشناخته به‌خودی‌خود FAIL نیست؛ چون Policy عمداً آن را Preserve می‌کند. چنین مواردی باید جداگانه گزارش شوند.
</p>

</div>

In [7]:
# Code Cell 07

changed_documents = 0
unchanged_documents = 0
changed_by_source = Counter()

empty_after_normalization = []
non_nfc_after_normalization = []
idempotence_failures = []

before_characters = 0
after_characters = 0

zwnj_before = 0
zwnj_after = 0

ascii_digits_before = 0
ascii_digits_after = 0

arabic_yeh_before = 0
arabic_yeh_after = 0

arabic_kaf_before = 0
arabic_kaf_after = 0

double_space_runs_before = 0
double_space_runs_after = 0

residual_required_cleanup = Counter()
preserved_unknown_controls = Counter()

control_rules = normalization_policy["transformations"]["control_characters"]
digit_mapping = normalization_policy["transformations"]["digit_mapping"]
soft_hyphen_character = (
    normalization_policy["transformations"]["soft_hyphen"]["character"]
)

records_dry_run = 0

with dataset_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)

        document_id = record["document_id"]
        source_type = record["source_type"]
        original_text = record["text"]

        normalized_text = normalize_persian_text(original_text)

        records_dry_run += 1

        before_characters += len(original_text)
        after_characters += len(normalized_text)

        zwnj_before += original_text.count("\u200c")
        zwnj_after += normalized_text.count("\u200c")

        ascii_digits_before += sum(
            original_text.count(character)
            for character in "0123456789"
        )
        ascii_digits_after += sum(
            normalized_text.count(character)
            for character in "0123456789"
        )

        arabic_yeh_before += original_text.count("\u064a")
        arabic_yeh_after += normalized_text.count("\u064a")

        arabic_kaf_before += original_text.count("\u0643")
        arabic_kaf_after += normalized_text.count("\u0643")

        double_space_runs_before += len(
            re.findall(r" {2,}", original_text)
        )
        double_space_runs_after += len(
            re.findall(r" {2,}", normalized_text)
        )

        if normalized_text != original_text:
            changed_documents += 1
            changed_by_source[source_type] += 1
        else:
            unchanged_documents += 1

        if original_text.strip() and not normalized_text.strip():
            empty_after_normalization.append(document_id)

        if not unicodedata.is_normalized("NFC", normalized_text):
            non_nfc_after_normalization.append(document_id)

        if normalize_persian_text(normalized_text) != normalized_text:
            idempotence_failures.append(document_id)

        if soft_hyphen_character in normalized_text:
            residual_required_cleanup["Soft hyphen U+00AD"] += (
                normalized_text.count(soft_hyphen_character)
            )

        if "\u00a0" in normalized_text:
            residual_required_cleanup["NBSP U+00A0"] += (
                normalized_text.count("\u00a0")
            )

        remaining_arabic_digits = sum(
            normalized_text.count(character)
            for character in digit_mapping
        )

        if remaining_arabic_digits:
            residual_required_cleanup["Arabic-Indic digits"] += (
                remaining_arabic_digits
            )

        for index, character in enumerate(normalized_text):
            config = control_rules.get(character)

            if config is None:
                continue

            left = (
                normalized_text[index - 1]
                if index > 0
                else None
            )
            right = (
                normalized_text[index + 1]
                if index + 1 < len(normalized_text)
                else None
            )

            pattern = (
                classify_neighbor(left),
                classify_neighbor(right),
            )

            action = config["rules"].get(
                pattern,
                config["default_action"],
            )

            if action == "preserve":
                preserved_unknown_controls[
                    f"{config['label']} | "
                    f"{pattern[0]} -> {pattern[1]}"
                ] += 1
            else:
                residual_required_cleanup[
                    f"{config['label']} | "
                    f"{pattern[0]} -> {pattern[1]} | "
                    f"expected={action}"
                ] += 1

print("Records dry-run:", records_dry_run)
print("Changed documents:", changed_documents)
print("Unchanged documents:", unchanged_documents)

print("\nChanged documents by source:")
for source_type, count in sorted(changed_by_source.items()):
    print(f"  {source_type}: {count}")

print("\nCharacter totals:")
print("  Before:", before_characters)
print("  After:", after_characters)
print("  Delta:", after_characters - before_characters)

print("\nPreservation checks:")
print("  ZWNJ:", zwnj_before, "->", zwnj_after)
print("  ASCII digits:", ascii_digits_before, "->", ascii_digits_after)
print("  Arabic yeh:", arabic_yeh_before, "->", arabic_yeh_after)
print("  Arabic kaf:", arabic_kaf_before, "->", arabic_kaf_after)

print("\nWhitespace checks:")
print(
    "  Double-space runs:",
    double_space_runs_before,
    "->",
    double_space_runs_after,
)

print("\nPost-normalization validation:")
print("  Empty texts created:", len(empty_after_normalization))
print("  Non-NFC texts remaining:", len(non_nfc_after_normalization))
print("  Idempotence failures:", len(idempotence_failures))

print("\nResidual required cleanup:")
if residual_required_cleanup:
    for label, count in sorted(residual_required_cleanup.items()):
        print(f"  {label}: {count}")
else:
    print("  None")

print("\nPreserved unknown control contexts:")
if preserved_unknown_controls:
    for label, count in sorted(preserved_unknown_controls.items()):
        print(f"  {label}: {count}")
else:
    print("  None")

assert records_dry_run == record_count, (
    f"Dry-run record mismatch: "
    f"expected {record_count}, found {records_dry_run}"
)

assert changed_documents + unchanged_documents == record_count

assert not empty_after_normalization, (
    f"Normalization created empty texts: "
    f"{len(empty_after_normalization)}"
)

assert not non_nfc_after_normalization, (
    f"Non-NFC texts remain: "
    f"{len(non_nfc_after_normalization)}"
)

assert not idempotence_failures, (
    f"Idempotence failures: "
    f"{len(idempotence_failures)}"
)

assert not residual_required_cleanup, (
    f"Known normalization targets remain: "
    f"{dict(residual_required_cleanup)}"
)

assert zwnj_after >= zwnj_before, (
    f"ZWNJ count unexpectedly decreased: "
    f"{zwnj_before} -> {zwnj_after}"
)

assert ascii_digits_before == ascii_digits_after, (
    f"ASCII digits changed: "
    f"{ascii_digits_before} -> {ascii_digits_after}"
)

assert arabic_yeh_before == arabic_yeh_after, (
    f"Arabic yeh changed: "
    f"{arabic_yeh_before} -> {arabic_yeh_after}"
)

assert arabic_kaf_before == arabic_kaf_after, (
    f"Arabic kaf changed: "
    f"{arabic_kaf_before} -> {arabic_kaf_after}"
)

assert double_space_runs_after <= double_space_runs_before, (
    f"Double-space runs increased: "
    f"{double_space_runs_before} -> {double_space_runs_after}"
)

print(
    "\nNB08 CONTEXT-AWARE CORPUS "
    "NORMALIZATION DRY RUN: PASS"
)

Records dry-run: 19456
Changed documents: 263
Unchanged documents: 19193

Changed documents by source:
  book: 132
  transcript: 13
  website: 118

Character totals:
  Before: 3125877
  After: 3125699
  Delta: -178

Preservation checks:
  ZWNJ: 59256 -> 59493
  ASCII digits: 6883 -> 6883
  Arabic yeh: 137 -> 137
  Arabic kaf: 45 -> 45

Whitespace checks:
  Double-space runs: 156 -> 156

Post-normalization validation:
  Empty texts created: 0
  Non-NFC texts remaining: 0
  Idempotence failures: 0

Residual required cleanup:
  None

Preserved unknown control contexts:
  None

NB08 CONTEXT-AWARE CORPUS NORMALIZATION DRY RUN: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Audit نهایی Ruleهای Context-aware</h3>

<p>
Dry Run کامل با Policy زمینه‌آگاه PASS شد. اکنون باید دقیقاً مشخص شود هر نوع Transformation چند بار اجرا شده و اثر نهایی آن بر طول متن و تعداد نیم‌فاصله‌ها چه بوده است.
</p>

<p>
این Audit Pipeline را به پنج Stage تقسیم می‌کند:
<code>NFC → Soft Hyphen → Contextual Controls → Contextual NBSP → Digit Mapping</code>.
</p>

<p>
برای Control Characterها نیز تعداد سه Action مستقل ثبت می‌شود:
<code>remove</code>،
<code>replace_with_zwnj</code>
و
<code>preserve</code>.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام <code>19456</code> سند Audit شوند.</li>
<li>خروجی Stage-by-Stage با تابع اصلی Normalizer در تمام اسناد یکسان باشد.</li>
<li>جمع Delta تمام Stageها دقیقاً برابر Delta Dry Run یعنی <code>-178</code> باشد.</li>
<li>افزایش تعداد ZWNJ دقیقاً با تعداد Actionهای <code>replace_with_zwnj</code> سازگار باشد.</li>
<li>هیچ Action ناشناخته‌ای اجرا نشده باشد.</li>
</ul>

<p>
این مرحله همچنان Read Only است و هیچ Artifactی روی Disk نوشته نمی‌شود.
</p>

</div>

In [8]:
# Code Cell 08

stage_changed_documents = Counter()
stage_character_delta = Counter()

control_action_occurrences = Counter()
control_action_documents = Counter()

stage_output_mismatches = []
unknown_actions = []

records_audited = 0

zwnj_created_by_rules = 0

control_rules = normalization_policy["transformations"]["control_characters"]
soft_hyphen_rule = normalization_policy["transformations"]["soft_hyphen"]
nbsp_rule = (
    normalization_policy["transformations"]
    ["contextual_rules"]
    ["nbsp_boundary"]
)
digit_mapping = normalization_policy["transformations"]["digit_mapping"]


def apply_contextual_control_rules_with_audit(text, policy):
    output = []
    action_counts = Counter()

    for index, character in enumerate(text):
        config = policy["transformations"]["control_characters"].get(character)

        if config is None:
            output.append(character)
            continue

        left = text[index - 1] if index > 0 else None
        right = text[index + 1] if index + 1 < len(text) else None

        pattern = (
            classify_neighbor(left),
            classify_neighbor(right),
        )

        action = config["rules"].get(
            pattern,
            config["default_action"],
        )

        audit_key = (
            config["label"],
            pattern[0],
            pattern[1],
            action,
        )

        action_counts[audit_key] += 1

        if action == "remove":
            continue

        if action == "replace_with_zwnj":
            output.append("\u200c")
            continue

        if action == "preserve":
            output.append(character)
            continue

        unknown_actions.append(
            {
                "label": config["label"],
                "pattern": pattern,
                "action": action,
            }
        )
        output.append(character)

    return "".join(output), action_counts


with dataset_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        document_id = record["document_id"]
        original = record["text"]

        records_audited += 1

        nfc_text = unicodedata.normalize(
            normalization_policy["unicode_form"],
            original,
        )

        if nfc_text != original:
            stage_changed_documents["NFC"] += 1

        stage_character_delta["NFC"] += (
            len(nfc_text) - len(original)
        )

        soft_hyphen_text = nfc_text

        if soft_hyphen_rule["action"] == "remove":
            soft_hyphen_text = soft_hyphen_text.replace(
                soft_hyphen_rule["character"],
                "",
            )

        if soft_hyphen_text != nfc_text:
            stage_changed_documents["Soft hyphen"] += 1

        stage_character_delta["Soft hyphen"] += (
            len(soft_hyphen_text) - len(nfc_text)
        )

        control_text, action_counts = (
            apply_contextual_control_rules_with_audit(
                soft_hyphen_text,
                normalization_policy,
            )
        )

        if control_text != soft_hyphen_text:
            stage_changed_documents["Contextual controls"] += 1

        stage_character_delta["Contextual controls"] += (
            len(control_text) - len(soft_hyphen_text)
        )

        for audit_key, count in action_counts.items():
            label, left_class, right_class, action = audit_key

            display_key = (
                f"{label} | "
                f"{left_class} -> {right_class} | "
                f"{action}"
            )

            control_action_occurrences[display_key] += count
            control_action_documents[display_key] += 1

            if action == "replace_with_zwnj":
                zwnj_created_by_rules += count

        before_nbsp = control_text
        nbsp = nbsp_rule["source"]

        if nbsp_rule["collapse_spacing_run_containing_nbsp"]:
            after_nbsp = re.sub(
                rf"[ {re.escape(nbsp)}]*"
                rf"{re.escape(nbsp)}"
                rf"[ {re.escape(nbsp)}]*",
                nbsp_rule["target"],
                before_nbsp,
            )
        else:
            after_nbsp = before_nbsp.replace(
                nbsp,
                nbsp_rule["target"],
            )

        if after_nbsp != before_nbsp:
            stage_changed_documents["Contextual NBSP"] += 1

        stage_character_delta["Contextual NBSP"] += (
            len(after_nbsp) - len(before_nbsp)
        )

        digit_text = after_nbsp

        for source, target in digit_mapping.items():
            digit_text = digit_text.replace(
                source,
                target,
            )

        if digit_text != after_nbsp:
            stage_changed_documents["Digit mapping"] += 1

        stage_character_delta["Digit mapping"] += (
            len(digit_text) - len(after_nbsp)
        )

        canonical_output = normalize_persian_text(original)

        if digit_text != canonical_output:
            stage_output_mismatches.append(document_id)


total_stage_delta = sum(stage_character_delta.values())
expected_total_delta = after_characters - before_characters

actual_zwnj_delta = zwnj_after - zwnj_before

print("Records audited:", records_audited)

print("\nStage impact:")
for stage in [
    "NFC",
    "Soft hyphen",
    "Contextual controls",
    "Contextual NBSP",
    "Digit mapping",
]:
    print(
        f"  {stage}: "
        f"changed_documents={stage_changed_documents[stage]}, "
        f"character_delta={stage_character_delta[stage]}"
    )

print("\nContextual control actions:")
for label, count in sorted(control_action_occurrences.items()):
    print(
        f"  {label}: "
        f"occurrences={count}, "
        f"documents={control_action_documents[label]}"
    )

print("\nZWNJ reconciliation:")
print("  ZWNJ before:", zwnj_before)
print("  ZWNJ after:", zwnj_after)
print("  Actual ZWNJ delta:", actual_zwnj_delta)
print("  replace_with_zwnj actions:", zwnj_created_by_rules)

print("\nDelta reconciliation:")
print("  Sum of stage deltas:", total_stage_delta)
print("  Dry-run delta:", expected_total_delta)
print("  Stage output mismatches:", len(stage_output_mismatches))
print("  Unknown actions:", len(unknown_actions))

assert records_audited == record_count, (
    f"Audit record mismatch: "
    f"expected {record_count}, found {records_audited}"
)

assert not stage_output_mismatches, (
    f"Stage output mismatches: "
    f"{len(stage_output_mismatches)}"
)

assert not unknown_actions, (
    f"Unknown control actions encountered: "
    f"{len(unknown_actions)}"
)

assert total_stage_delta == expected_total_delta, (
    f"Character delta mismatch: "
    f"stages={total_stage_delta}, dry_run={expected_total_delta}"
)

assert actual_zwnj_delta == zwnj_created_by_rules, (
    f"ZWNJ reconciliation failed: "
    f"delta={actual_zwnj_delta}, "
    f"created_by_rules={zwnj_created_by_rules}"
)

print(
    "\nNB08 CONTEXT-AWARE TRANSFORMATION AUDIT: PASS"
)

Records audited: 19456

Stage impact:
  NFC: changed_documents=33, character_delta=-18
  Soft hyphen: changed_documents=0, character_delta=0
  Contextual controls: changed_documents=169, character_delta=-63
  Contextual NBSP: changed_documents=73, character_delta=-97
  Digit mapping: changed_documents=2, character_delta=0

Contextual control actions:
  BOM U+FEFF | LETTER -> LETTER | replace_with_zwnj: occurrences=205, documents=107
  LRM U+200E | LETTER -> LETTER | replace_with_zwnj: occurrences=32, documents=27
  LRM U+200E | LETTER -> PUNCT | remove: occurrences=1, documents=1
  LRM U+200E | LETTER -> SPACE | remove: occurrences=3, documents=2
  LRM U+200E | LETTER -> ZWNJ | remove: occurrences=25, documents=19
  LRM U+200E | SPACE -> LETTER | remove: occurrences=2, documents=2
  LRM U+200E | ZWNJ -> LETTER | remove: occurrences=9, documents=9
  RLM U+200F | LETTER -> ZWNJ | remove: occurrences=7, documents=4
  RLM U+200F | MARK -> SPACE | remove: occurrences=2, documents=1
  RLM U+

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Root Cause Inspection — تفکیک کاربرد فارسی و عربیِ ي / ك</h3>

<p>
بازبینی انسانی نشان داد حفظ سراسری <code>ي</code> و <code>ك</code> نیز برای کل Corpus مناسب نیست.
این نویسه‌ها هم در نقل‌قول‌های عربی واقعی دیده می‌شوند و هم در نثر فارسیِ دارای Encoding قدیمی؛ مانند <code>بدهكار</code>، <code>دوري</code> و <code>پايان</code>.
</p>

<p>
بنابراین هنوز هیچ تبدیل جدیدی اعمال نمی‌شود. ابتدا تمام occurrenceهای واقعی <code>ي</code> و <code>ك</code> بررسی و بر اساس Context قابل مشاهده دسته‌بندی می‌شوند.
</p>

<p>
این Inspection موارد زیر را اندازه‌گیری می‌کند:</p>

<ul>
<li>تعداد occurrence و Document coverage برای هر نویسه؛</li>
<li>واژه‌ای که نویسه داخل آن قرار دارد؛</li>
<li>وجود نشانه‌های قوی متن عربی مانند حرکات Unicode در Context؛</li>
<li>وجود حروف مشخص فارسی مانند <code>پ، چ، ژ، گ</code> در Context؛</li>
<li>توزیع Sourceها و نمونه‌های واقعی هر گروه.</li>
</ul>

<p>
این دسته‌بندی صرفاً Evidence برای طراحی Rule بعدی است و خودش به معنی تشخیص قطعی زبان یا تغییر متن نیست.
</p>

<h4>معیار این مرحله</h4>

<p>
تمام occurrenceهای <code>ي/ك</code> از Corpus واقعی پوشش داده شوند و Context کافی برای تصمیم‌گیری بعدی فراهم شود. هیچ Artifact یا Datasetی تغییر نمی‌کند.
</p>

</div>

In [9]:
# Code Cell 09

arabic_form_targets = {
    "Arabic yeh U+064A": "\u064a",
    "Arabic kaf U+0643": "\u0643",
}

persian_specific_letters = set("پچژگ")
arabic_specific_letters = set("ىةؤئأإآء")

context_radius = 55
max_samples_per_bucket = 5

occurrence_counts = Counter()
document_coverage = {
    label: set()
    for label in arabic_form_targets
}
source_counts = {
    label: Counter()
    for label in arabic_form_targets
}
bucket_counts = {
    label: Counter()
    for label in arabic_form_targets
}
token_counts = {
    label: Counter()
    for label in arabic_form_targets
}
bucket_samples = {
    label: {}
    for label in arabic_form_targets
}


def is_token_character(character):
    if character == "\u200c":
        return True

    category = unicodedata.category(character)

    return (
        category.startswith("L")
        or category.startswith("M")
    )


def extract_token(text, index):
    start = index
    end = index + 1

    while start > 0 and is_token_character(text[start - 1]):
        start -= 1

    while end < len(text) and is_token_character(text[end]):
        end += 1

    return text[start:end]


records_inspected = 0

with dataset_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        text = record["text"]
        document_id = record["document_id"]
        source_type = record["source_type"]

        records_inspected += 1

        for label, target in arabic_form_targets.items():
            search_start = 0

            while True:
                index = text.find(target, search_start)

                if index == -1:
                    break

                occurrence_counts[label] += 1
                document_coverage[label].add(document_id)
                source_counts[label][source_type] += 1

                token = extract_token(text, index)
                token_counts[label][token] += 1

                start = max(0, index - context_radius)
                end = min(
                    len(text),
                    index + context_radius + 1,
                )

                context = text[start:end]

                has_combining_marks = any(
                    unicodedata.category(character).startswith("M")
                    for character in context
                )

                has_persian_specific = any(
                    character in persian_specific_letters
                    for character in context
                )

                has_arabic_specific = any(
                    character in arabic_specific_letters
                    for character in context
                )

                bucket = (
                    f"marks={has_combining_marks} | "
                    f"persian_specific={has_persian_specific} | "
                    f"arabic_specific={has_arabic_specific}"
                )

                bucket_counts[label][bucket] += 1

                samples = bucket_samples[label].setdefault(
                    bucket,
                    [],
                )

                if len(samples) < max_samples_per_bucket:
                    samples.append(
                        {
                            "document_id": document_id,
                            "source_type": source_type,
                            "token": token,
                            "context": context,
                        }
                    )

                search_start = index + 1


print("Records inspected:", records_inspected)

for label in arabic_form_targets:
    print(f"\n=== {label} ===")
    print("Occurrences:", occurrence_counts[label])
    print(
        "Documents:",
        len(document_coverage[label]),
    )

    print("\nSource distribution:")
    for source_type, count in sorted(
        source_counts[label].items()
    ):
        print(f"  {source_type}: {count}")

    print("\nMost frequent tokens:")
    for token, count in token_counts[label].most_common(25):
        print(
            f"  {repr(token)}: {count}"
        )

    print("\nContext buckets:")

    for bucket, count in bucket_counts[label].most_common():
        print(f"\n{bucket}")
        print("Count:", count)

        for sample in bucket_samples[label][bucket]:
            print(
                f"  [{sample['source_type']}] "
                f"{sample['document_id']}"
            )
            print(
                "    token:",
                repr(sample["token"]),
            )
            print(
                "    context:",
                repr(sample["context"]),
            )


expected_occurrences = {
    "Arabic yeh U+064A": 137,
    "Arabic kaf U+0643": 45,
}

print("\nCoverage reconciliation:")

for label, expected in expected_occurrences.items():
    actual = occurrence_counts[label]

    print(
        f"  {label}: "
        f"expected={expected}, actual={actual}"
    )

    assert actual == expected, (
        f"{label} occurrence mismatch: "
        f"expected {expected}, found {actual}"
    )

assert records_inspected == record_count, (
    f"Inspection record mismatch: "
    f"expected {record_count}, found {records_inspected}"
)

print(
    "\nNB08 ARABIC/PERSIAN YEH-KAF "
    "ROOT CAUSE INSPECTION: COMPLETE"
)

Records inspected: 19456

=== Arabic yeh U+064A ===
Occurrences: 137
Documents: 41

Source distribution:
  book: 90
  transcript: 42
  website: 5

Most frequent tokens:
  'يا': 6
  'خريد': 4
  'ريسک\u200cهای': 4
  'توليد': 3
  'فِي': 3
  'تجهيز': 2
  'كارگيري': 2
  'نيست': 2
  'پايان': 2
  'سررسيد': 2
  'قيمت': 2
  'هزينه\u200cهای': 2
  'ساير': 2
  'پيش\u200cخريدِ': 2
  'بين': 2
  'فيزيكی': 2
  'سياستهاي': 2
  'وَالَّذِينَ': 2
  'الَّذِينَ': 2
  'وَفِي': 2
  'الشَّيَاطِينِ': 2
  'سيأتيك': 2
  'دوري': 1
  'زيرا': 1
  'وسائل\u200cالشيعه': 1

Context buckets:

marks=False | persian_specific=True | arabic_specific=True
Count: 38
  [website] website:B4673E14
    token: 'زيرا'
    context: 'از بدهكار شدن دوري كن؛ زيرا بدهکاری عیب انسان دین\u200cدار است.\nپیامبر خدا (ص)\nوسائل\u200c'
  [website] website:B4673E14
    token: 'وسائل\u200cالشيعه'
    context: 'بدهکاری عیب انسان دین\u200cدار است.\nپیامبر خدا (ص)\nوسائل\u200cالشيعه، ج\u200c۱۸، ص۳۱۵\nپیامبر خدا (ص) فرمودند: «به خدا پناه مى\u200f'
  

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تصمیم نهایی درباره ي / ك — Preserve و Deferred</h3>

<p>
Inspection کامل Corpus نشان داد نویسه‌های عربی <code>ي</code> و <code>ك</code>
در دو نوع محتوای متفاوت حضور دارند:
</p>

<ul>
<li>نثر فارسی با Orthography یا Encoding قدیمی؛ مانند <code>پايان</code>، <code>خريد</code>، <code>بدهكار</code> و <code>بانكی</code>.</li>
<li>نقل‌قول‌ها و عبارت‌های عربی واقعی؛ مانند آیات و متون دارای حرکات عربی.</li>
</ul>

<p>
ویژگی‌های سطح Character و Context برای جداسازی قطعی این دو گروه کافی نیستند.
حتی Bucketهای دارای حرکات عربی یا حروف اختصاصی فارسی نیز شامل نمونه‌های مخلوط هستند.
</p>

<p>
بنابراین تبدیل سراسری یا Heuristic-based نویسه‌های <code>ي/ك</code> در NB08
ریسک تغییر غیرمجاز متن عربی دارد.
</p>

<h4>Policy نهایی NB08</h4>

<ul>
<li><code>ي U+064A</code> بدون تغییر حفظ می‌شود.</li>
<li><code>ك U+0643</code> بدون تغییر حفظ می‌شود.</li>
<li>تبدیل سراسری آن‌ها به <code>ی/ک</code> ممنوع است.</li>
<li>پاک‌سازی احتمالی این موارد به مرحله‌ای دارای تشخیص قابل اتکای زبان، Quote Boundary یا Human-reviewed mapping موکول می‌شود.</li>
<li>وجود این نویسه‌ها در خروجی NB08 Error محسوب نمی‌شود؛ اما تعداد و Coverage آن‌ها باید Audit شود.</li>
</ul>

<h4>معیار PASS</h4>

<p>
تعداد <code>ي/ك</code> در سطح کل Corpus و در سطح هر Document باید قبل و بعد از Normalization یکسان باقی بماند و نمونه‌های فارسی و عربی هر دو در Audit قابل مشاهده باشند.
</p>

</div>

In [10]:
# Code Cell 10

target_characters = {
    "Arabic yeh U+064A": "\u064a",
    "Arabic kaf U+0643": "\u0643",
}

global_before = Counter()
global_after = Counter()

document_count_mismatches = []

persian_mixed_samples = []
arabic_quoted_samples = []

max_samples = 4
sample_radius = 60

records_checked = 0

with dataset_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)

        document_id = record["document_id"]
        source_type = record["source_type"]
        original = record["text"]
        normalized = normalize_persian_text(original)

        records_checked += 1

        contains_target = False

        for label, character in target_characters.items():
            before_count = original.count(character)
            after_count = normalized.count(character)

            global_before[label] += before_count
            global_after[label] += after_count

            if before_count:
                contains_target = True

            if before_count != after_count:
                document_count_mismatches.append(
                    {
                        "document_id": document_id,
                        "label": label,
                        "before": before_count,
                        "after": after_count,
                    }
                )

        if not contains_target:
            continue

        target_indexes = [
            index
            for index, character in enumerate(original)
            if character in {"\u064a", "\u0643"}
        ]

        if not target_indexes:
            continue

        target_index = target_indexes[0]

        start = max(
            0,
            target_index - sample_radius,
        )
        end = min(
            len(original),
            target_index + sample_radius + 1,
        )

        context = original[start:end]
        normalized_context = normalized[
            start:min(len(normalized), end + 10)
        ]

        has_combining_marks = any(
            unicodedata.category(character).startswith("M")
            for character in context
        )

        has_persian_specific = any(
            character in set("پچژگ")
            for character in context
        )

        sample = {
            "document_id": document_id,
            "source_type": source_type,
            "before": context,
            "after": normalized_context,
        }

        if (
            len(persian_mixed_samples) < max_samples
            and source_type in {"book", "website"}
            and has_persian_specific
        ):
            persian_mixed_samples.append(sample)

        if (
            len(arabic_quoted_samples) < max_samples
            and source_type == "transcript"
            and has_combining_marks
        ):
            arabic_quoted_samples.append(sample)


print("Records checked:", records_checked)

print("\nGlobal preservation:")

for label in target_characters:
    print(
        f"  {label}: "
        f"{global_before[label]} -> "
        f"{global_after[label]}"
    )

print(
    "\nPer-document count mismatches:",
    len(document_count_mismatches),
)

if document_count_mismatches:
    for mismatch in document_count_mismatches[:10]:
        print(
            json.dumps(
                mismatch,
                ensure_ascii=False,
                indent=2,
            )
        )

print(
    "\n=== Persian / mixed legacy orthography samples ==="
)

for index, sample in enumerate(
    persian_mixed_samples,
    start=1,
):
    print(f"\nSample {index}")
    print("Document ID:", sample["document_id"])
    print("Source:", sample["source_type"])
    print("BEFORE:", repr(sample["before"]))
    print("AFTER :", repr(sample["after"]))

print(
    "\n=== Arabic / marked quotation samples ==="
)

for index, sample in enumerate(
    arabic_quoted_samples,
    start=1,
):
    print(f"\nSample {index}")
    print("Document ID:", sample["document_id"])
    print("Source:", sample["source_type"])
    print("BEFORE:", repr(sample["before"]))
    print("AFTER :", repr(sample["after"]))

print("\nDecision checks:")
print(
    "  Global Arabic yeh conversion forbidden:",
    "convert_arabic_yeh_globally"
    in normalization_policy["forbidden_operations"],
)
print(
    "  Global Arabic kaf conversion forbidden:",
    "convert_arabic_kaf_globally"
    in normalization_policy["forbidden_operations"],
)
print(
    "  Persian/mixed samples:",
    len(persian_mixed_samples),
)
print(
    "  Arabic/quoted samples:",
    len(arabic_quoted_samples),
)

assert records_checked == record_count

assert global_before["Arabic yeh U+064A"] == 137
assert global_before["Arabic kaf U+0643"] == 45

assert global_before == global_after, (
    f"Arabic yeh/kaf global counts changed: "
    f"before={dict(global_before)}, "
    f"after={dict(global_after)}"
)

assert not document_count_mismatches, (
    f"Arabic yeh/kaf changed inside "
    f"{len(document_count_mismatches)} documents"
)

assert (
    "convert_arabic_yeh_globally"
    in normalization_policy["forbidden_operations"]
)

assert (
    "convert_arabic_kaf_globally"
    in normalization_policy["forbidden_operations"]
)

assert persian_mixed_samples, (
    "No Persian/mixed legacy samples found."
)

assert arabic_quoted_samples, (
    "No Arabic/quoted samples found."
)

print(
    "\nNB08 YEH/KAF PRESERVE + DEFER DECISION: PASS"
)

Records checked: 19456

Global preservation:
  Arabic yeh U+064A: 137 -> 137
  Arabic kaf U+0643: 45 -> 45

Per-document count mismatches: 0

=== Persian / mixed legacy orthography samples ===

Sample 1
Document ID: website:B4673E14
Source: website
BEFORE: 'از بدهكار شدن دوري كن؛ زيرا بدهکاری عیب انسان دین\u200cدار است.\nپیامبر خ'
AFTER : 'از بدهكار شدن دوري كن؛ زيرا بدهکاری عیب انسان دین\u200cدار است.\nپیامبر خدا (ص)\nوسا'

Sample 2
Document ID: website:FD1A77EC
Source: website
BEFORE: 'امام صادق (ع): خوب است پیش از آنكه درخواستی كنی، هدیه بدهی. سه نوع هدیه وجود دارد: هدیه\u200cای كه'
AFTER : 'امام صادق (ع): خوب است پیش از آنكه درخواستی كنی، هدیه بدهی. سه نوع هدیه وجود دارد: هدیه\u200cای كه در برابرش'

Sample 3
Document ID: book:adult-financial-literacy-course:fa-source:p000026
Source: book
BEFORE: 'تجهيز منابع بانكی با سپرده\u200cگذاری16'
AFTER : 'تجهيز منابع بانكی با سپرده\u200cگذاری16'

Sample 4
Document ID: book:adult-financial-literacy-course:fa-source:p000338
Source: book
BEFORE

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery قرارداد Artifact و Output برای NB08</h3>

<p>
Normalization Policy و Normalizer اکنون تثبیت شده‌اند. قبل از ساخت هر Artifact باید قرارداد خروجی موجود در Repository کشف شود.
</p>

<p>
اجرای اولیه این مرحله به دلیل اتکا به متغیر <code>repo_root</code> که در Kernel فعلی تعریف نشده بود متوقف شد. در نسخه اصلاح‌شده، Repository Root مستقیماً از Path واقعی Dataset و وجود <code>.git</code> کشف می‌شود و هیچ Path حدس‌زده نمی‌شود.
</p>

<p>
در این مرحله هیچ فایل یا Schema جدیدی ساخته یا تغییر داده نمی‌شود.
</p>

<h4>موارد مورد بررسی</h4>

<ul>
<li>Repository Root واقعی؛</li>
<li>ساختار پوشه Processed مربوط به Dataset ورودی؛</li>
<li>Schemaهای موجود Phase 2؛</li>
<li>قواعد Schema فعلی درباره Propertyهای اضافی؛</li>
<li>وجود یا عدم وجود <code>raw_text</code>، <code>normalized_text</code> و <code>transformation_log</code>؛</li>
<li>Referenceهای موجود Repository به Normalization و Artifactهای مشابه؛</li>
<li>Conventionهای فعلی نام‌گذاری و Versioning.</li>
</ul>

<h4>معیار این مرحله</h4>

<p>
این Cell فقط Discovery است. نتیجه آن مشخص می‌کند قرارداد Artifact خروجی NB08 چگونه باید تعریف شود. هیچ Dataset، Schema یا Artifactی نوشته نمی‌شود.
</p>

</div>

In [11]:
# Code Cell 11

from pathlib import Path
import json


def human_size(size):
    units = ["B", "KB", "MB", "GB"]
    value = float(size)

    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1024


# Repository root را از Path واقعی Dataset کشف می‌کنیم.
dataset_path = Path(dataset_path).resolve()
schema_path = Path(schema_path).resolve()

repo_root_candidates = [
    dataset_path.parent,
    *dataset_path.parents,
]

repo_root = next(
    (
        candidate
        for candidate in repo_root_candidates
        if (candidate / ".git").exists()
    ),
    None,
)

assert repo_root is not None, (
    "Repository root could not be discovered "
    "from dataset_path by locating .git"
)

assert dataset_path.exists(), (
    f"Dataset does not exist: {dataset_path}"
)

assert schema_path.exists(), (
    f"Schema does not exist: {schema_path}"
)


print("Repository root:", repo_root)
print("Input dataset:", dataset_path)
print("Input schema:", schema_path)


phase2_data_root = dataset_path.parents[1]
processed_root = dataset_path.parent
schema_root = schema_path.parent

print("\nDerived existing roots:")
print("  Phase2 data root:", phase2_data_root)
print("  Processed root:", processed_root)
print("  Schema root:", schema_root)

assert phase2_data_root.exists()
assert processed_root.exists()
assert schema_root.exists()


print("\n=== Existing processed artifacts ===")

processed_files = sorted(
    path
    for path in processed_root.iterdir()
    if path.is_file()
)

for path in processed_files:
    print(
        f"  {path.relative_to(repo_root)} "
        f"| {human_size(path.stat().st_size)}"
    )


print("\n=== Existing Phase2 schemas ===")

schema_files = sorted(
    path
    for path in schema_root.iterdir()
    if path.is_file()
)

for path in schema_files:
    print(
        f"  {path.relative_to(repo_root)} "
        f"| {human_size(path.stat().st_size)}"
    )


with schema_path.open("r", encoding="utf-8") as handle:
    current_schema = json.load(handle)

schema_properties = current_schema.get(
    "properties",
    {},
)

schema_required = current_schema.get(
    "required",
    [],
)

schema_additional = current_schema.get(
    "additionalProperties",
    "<not specified>",
)

candidate_output_fields = [
    "text",
    "raw_text",
    "normalized_text",
    "normalization",
    "normalization_policy",
    "transformation_log",
]


print("\n=== Current common schema contract ===")
print("Title:", current_schema.get("title"))
print("$id:", current_schema.get("$id"))
print("additionalProperties:", schema_additional)
print("Property count:", len(schema_properties))
print("Required field count:", len(schema_required))

print("\nCandidate normalization fields in current schema:")

for field in candidate_output_fields:
    print(
        f"  {field}:",
        (
            "PRESENT"
            if field in schema_properties
            else "ABSENT"
        ),
    )

if "text" in schema_properties:
    print("\nCurrent 'text' property:")
    print(
        json.dumps(
            schema_properties["text"],
            ensure_ascii=False,
            indent=2,
        )
    )


search_terms = [
    "raw_text",
    "normalized_text",
    "transformation_log",
    "normalization_policy",
    "persian_normalization",
    "normalize_persian",
]

searchable_suffixes = {
    ".py",
    ".json",
    ".yaml",
    ".yml",
    ".md",
    ".ipynb",
    ".toml",
}

excluded_parts = {
    ".git",
    ".venv",
    "venv",
    "node_modules",
    "__pycache__",
}

max_file_size = 2 * 1024 * 1024
max_hits_per_term = 15

repository_hits = {
    term: []
    for term in search_terms
}


for path in repo_root.rglob("*"):
    if not path.is_file():
        continue

    relative = path.relative_to(repo_root)

    if any(
        part in excluded_parts
        for part in relative.parts
    ):
        continue

    # Datasetهای حجیم وارد Text Search نمی‌شوند.
    if "data" in relative.parts:
        continue

    if path.suffix.lower() not in searchable_suffixes:
        continue

    if path.stat().st_size > max_file_size:
        continue

    try:
        content = path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    except Exception:
        continue

    lowered = content.lower()

    for term in search_terms:
        if (
            len(repository_hits[term])
            >= max_hits_per_term
        ):
            continue

        search_from = 0
        term_lower = term.lower()

        while (
            len(repository_hits[term])
            < max_hits_per_term
        ):
            index = lowered.find(
                term_lower,
                search_from,
            )

            if index == -1:
                break

            line_number = (
                content.count(
                    "\n",
                    0,
                    index,
                )
                + 1
            )

            line_start = (
                content.rfind(
                    "\n",
                    0,
                    index,
                )
                + 1
            )

            line_end = content.find(
                "\n",
                index,
            )

            if line_end == -1:
                line_end = len(content)

            excerpt = content[
                line_start:line_end
            ].strip()

            if len(excerpt) > 220:
                excerpt = excerpt[:220] + "..."

            repository_hits[term].append(
                {
                    "path": str(relative),
                    "line": line_number,
                    "excerpt": excerpt,
                }
            )

            search_from = (
                index + len(term_lower)
            )


print(
    "\n=== Repository contract/reference discovery ==="
)

for term in search_terms:
    hits = repository_hits[term]

    print(
        f"\nTerm: {term} | "
        f"hits shown={len(hits)}"
    )

    if not hits:
        print("  None")
        continue

    for hit in hits:
        print(
            f"  {hit['path']}:{hit['line']}"
        )
        print(
            f"    {hit['excerpt']}"
        )


print("\n=== Discovery summary ===")

print(
    "Current schema can accept undeclared fields:",
    schema_additional is not False,
)

print(
    "raw_text already defined:",
    "raw_text" in schema_properties,
)

print(
    "normalized_text already defined:",
    "normalized_text" in schema_properties,
)

print(
    "transformation_log already defined:",
    "transformation_log" in schema_properties,
)

print(
    "\nNB08 OUTPUT CONTRACT DISCOVERY: COMPLETE"
)

Repository root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Input dataset: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/processed/documents_unified_v1.jsonl
Input schema: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/schemas/phase2/document.schema.json

Derived existing roots:
  Phase2 data root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2
  Processed root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/processed
  Schema root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/schemas/phase2

=== Existing processed artifacts ===
  data/phase2/processed/documents_normalized_v1.jsonl | 60.85 MB
  data/phase2/processed/documents_unified_v1.jsonl | 60.85 MB
  data/phase2/processed/persian_normalization_records_v1.jsonl | 23.92 MB
  data/phase2/processed/website_citation_locators_v1.json |

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery تکمیلی قرارداد Hash، Report و Checkpoint</h3>

<p>
Schema فعلی Common Document بسته است و اجازه افزودن فیلدهای Normalization را نمی‌دهد. بنابراین خروجی NB08 احتمالاً باید از دو لایه تشکیل شود:
</p>

<ul>
<li>یک Dataset قابل مصرف توسط مرحله بعد که Common Document معتبر باقی بماند؛</li>
<li>یک Artifact جانبی برای نگهداری هم‌زمان Raw/Normalized Text و Transformation Log.</li>
</ul>

<p>
اما قبل از تعریف این قرارداد، باید semantics دقیق Hashهای سند و Convention واقعی Report/Checkpoint در Repository بررسی شود.
</p>

<h4>موارد Discovery</h4>

<ul>
<li>تعریف دقیق <code>text_sha256</code> و <code>raw_content_hash</code> در Schema؛</li>
<li>تعریف <code>schema_version</code> و محدودیت‌های آن؛</li>
<li>نمونه واقعی یک Common Document و رابطه Hash با <code>text</code>؛</li>
<li>ساختار فعلی <code>reports/phase2</code>؛</li>
<li>ساختار فعلی <code>artifacts/phase2/checkpoints</code>؛</li>
<li>Convention فایل‌های Checkpoint موجود؛</li>
</ul>

<p>
این مرحله Read Only است. هیچ Dataset، Schema، Report یا Checkpoint جدیدی نوشته نمی‌شود.
</p>

</div>

In [12]:
# Code Cell 12

import hashlib
import json
from pathlib import Path


# Re-discover critical paths from known input artifacts.
dataset_path = Path(dataset_path).resolve()
schema_path = Path(schema_path).resolve()

repo_root = next(
    (
        candidate
        for candidate in [
            dataset_path.parent,
            *dataset_path.parents,
        ]
        if (candidate / ".git").exists()
    ),
    None,
)

assert repo_root is not None

reports_phase2_root = repo_root / "reports" / "phase2"
checkpoints_root = (
    repo_root
    / "artifacts"
    / "phase2"
    / "checkpoints"
)


with schema_path.open("r", encoding="utf-8") as handle:
    current_schema = json.load(handle)

properties = current_schema["properties"]


print("=== Hash / version schema contract ===")

for field in [
    "schema_version",
    "text_sha256",
    "raw_content_hash",
    "document_id",
    "source_id",
]:
    print(f"\nField: {field}")

    field_schema = properties.get(field)

    if field_schema is None:
        print("  ABSENT")
    else:
        print(
            json.dumps(
                field_schema,
                ensure_ascii=False,
                indent=2,
            )
        )


print("\n=== First Common Document hash verification ===")

first_record = None

with dataset_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            first_record = json.loads(line)
            break

assert first_record is not None

computed_text_sha256 = hashlib.sha256(
    first_record["text"].encode("utf-8")
).hexdigest()

print("Document ID:", first_record["document_id"])
print("Schema version:", first_record["schema_version"])
print("Stored text_sha256:", first_record["text_sha256"])
print("Computed text_sha256:", computed_text_sha256)
print(
    "text_sha256 matches text:",
    first_record["text_sha256"] == computed_text_sha256,
)

print(
    "raw_content_hash:",
    first_record["raw_content_hash"],
)

print(
    "raw_content_hash equals text_sha256:",
    first_record["raw_content_hash"]
    == first_record["text_sha256"],
)

assert (
    first_record["text_sha256"]
    == computed_text_sha256
), (
    "Stored text_sha256 does not match "
    "SHA256(text) in first record."
)


print("\n=== reports/phase2 structure ===")

if reports_phase2_root.exists():
    report_files = sorted(
        path
        for path in reports_phase2_root.rglob("*")
        if path.is_file()
    )

    print("Report files:", len(report_files))

    for path in report_files[:80]:
        print(
            " ",
            path.relative_to(repo_root),
        )
else:
    print("reports/phase2: ABSENT")


print("\n=== checkpoint structure ===")

if checkpoints_root.exists():
    checkpoint_files = sorted(
        path
        for path in checkpoints_root.rglob("*")
        if path.is_file()
    )

    print("Checkpoint files:", len(checkpoint_files))

    for path in checkpoint_files[:50]:
        print(
            " ",
            path.relative_to(repo_root),
        )
else:
    checkpoint_files = []
    print("artifacts/phase2/checkpoints: ABSENT")


print("\n=== Existing checkpoint contract samples ===")

checkpoint_samples = []

for path in checkpoint_files[:10]:
    try:
        payload = json.loads(
            path.read_text(encoding="utf-8")
        )
    except Exception as exc:
        print(
            f"\n{path.relative_to(repo_root)} "
            f"| unreadable JSON: {exc}"
        )
        continue

    checkpoint_samples.append(
        {
            "path": path,
            "payload": payload,
        }
    )

    print(
        f"\n{path.relative_to(repo_root)}"
    )

    if isinstance(payload, dict):
        print(
            "Keys:",
            sorted(payload.keys()),
        )

        for key in [
            "notebook",
            "status",
            "version",
            "input",
            "inputs",
            "output",
            "outputs",
            "metrics",
            "hashes",
        ]:
            if key in payload:
                value = payload[key]

                serialized = json.dumps(
                    value,
                    ensure_ascii=False,
                )

                if len(serialized) > 700:
                    serialized = (
                        serialized[:700]
                        + "..."
                    )

                print(
                    f"  {key}: {serialized}"
                )
    else:
        print(
            "Top-level type:",
            type(payload).__name__,
        )


print("\n=== Discovery summary ===")
print(
    "Common schema additionalProperties:",
    current_schema.get(
        "additionalProperties",
        "<not specified>",
    ),
)
print(
    "text_sha256 verified against text:",
    True,
)
print(
    "reports/phase2 exists:",
    reports_phase2_root.exists(),
)
print(
    "checkpoints root exists:",
    checkpoints_root.exists(),
)
print(
    "checkpoint samples parsed:",
    len(checkpoint_samples),
)

print(
    "\nNB08 HASH + REPORT + CHECKPOINT "
    "CONTRACT DISCOVERY: COMPLETE"
)

=== Hash / version schema contract ===

Field: schema_version
{
  "const": "common_document_v1"
}

Field: text_sha256
{
  "pattern": "^[0-9a-f]{64}$",
  "type": "string"
}

Field: raw_content_hash
{
  "pattern": "^[0-9a-f]{64}$",
  "type": "string"
}

Field: document_id
{
  "minLength": 1,
  "type": "string"
}

Field: source_id
{
  "minLength": 1,
  "type": "string"
}

=== First Common Document hash verification ===
Document ID: website:008E61CC
Schema version: common_document_v1
Stored text_sha256: a4f3c022655386a29059ec10a797df3233a51d9590bb5958605e651136f118a2
Computed text_sha256: a4f3c022655386a29059ec10a797df3233a51d9590bb5958605e651136f118a2
text_sha256 matches text: True
raw_content_hash: 2510dbf81e13f55647cb9e89d4e1f2ecb6dc6276696f8e7871e1132ea5c940bd
raw_content_hash equals text_sha256: False

=== reports/phase2 structure ===
Report files: 13
  reports/phase2/book_ingestion_acceptance_v1.json
  reports/phase2/book_inventory_acceptance_v1.json
  reports/phase2/data_quality/web

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery نهایی قرارداد Report و Processing State</h3>

<p>
تا اینجا مشخص شده است که Dataset نهایی باید همچنان با <code>common_document_v1</code> معتبر بماند؛ بنابراین اگر فیلد <code>text</code> نرمال شود، <code>text_sha256</code> نیز باید دوباره محاسبه شود، در حالی که <code>raw_content_hash</code> به‌عنوان Hash منشأ خام حفظ می‌شود.
</p>

<p>
قبل از تعریف Artifactهای NB08، اکنون ساختار Reportهای واقعی NB07 و بخش‌های قابل استفاده Common Document بررسی می‌شوند.
</p>

<h4>اهداف</h4>

<ul>
<li>مشاهده Top-level Contract گزارش Acceptance و Report مرحله NB07؛</li>
<li>بررسی تعریف Schema برای <code>source_metadata</code>، <code>lineage</code> و <code>content_origin</code>؛</li>
<li>مشاهده نمونه واقعی از هر سه Source؛</li>
<li>کشف وجود یا عدم وجود <code>processing_state</code> در Common Documentهای نهایی؛</li>
<li>جلوگیری از افزودن Metadata جدید به مکان نامناسب.</li>
</ul>

<p>
این مرحله کاملاً Read Only است و هیچ فایل جدیدی نوشته نمی‌شود.
</p>

</div>

In [13]:
# Code Cell 13

from collections import Counter
from pathlib import Path
import json


reports_to_inspect = [
    repo_root
    / "reports"
    / "phase2"
    / "unified_schema_lineage_acceptance_v1.json",

    repo_root
    / "reports"
    / "phase2"
    / "unified_schema_lineage_report_v1.json",
]


print("=== NB07 report contract discovery ===")

for report_path in reports_to_inspect:
    print(
        f"\nReport: {report_path.relative_to(repo_root)}"
    )

    assert report_path.exists(), (
        f"Expected NB07 report missing: {report_path}"
    )

    payload = json.loads(
        report_path.read_text(encoding="utf-8")
    )

    print(
        "Top-level type:",
        type(payload).__name__,
    )

    if isinstance(payload, dict):
        print(
            "Top-level keys:",
            sorted(payload.keys()),
        )

        for key, value in payload.items():
            serialized = json.dumps(
                value,
                ensure_ascii=False,
            )

            if len(serialized) > 1000:
                serialized = (
                    serialized[:1000]
                    + "..."
                )

            print(
                f"  {key}: {serialized}"
            )


print("\n=== Relevant Common Document schema sections ===")

for field in [
    "source_metadata",
    "lineage",
    "content_origin",
    "attribution",
    "rights",
]:
    print(f"\nField: {field}")

    field_schema = schema_properties.get(field)

    assert field_schema is not None, (
        f"Schema field missing: {field}"
    )

    print(
        json.dumps(
            field_schema,
            ensure_ascii=False,
            indent=2,
        )
    )


print("\n=== Real document samples by source ===")

sample_by_source = {}
processing_state_paths = Counter()

with dataset_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        source_type = record["source_type"]

        if source_type not in sample_by_source:
            sample_by_source[source_type] = record

        # Discover processing_state recursively without assuming location.
        stack = [
            ("$", record),
        ]

        while stack:
            path, value = stack.pop()

            if isinstance(value, dict):
                for key, child in value.items():
                    child_path = f"{path}.{key}"

                    if key == "processing_state":
                        processing_state_paths[
                            child_path
                        ] += 1

                    stack.append(
                        (child_path, child)
                    )

            elif isinstance(value, list):
                for index, child in enumerate(value):
                    stack.append(
                        (
                            f"{path}[{index}]",
                            child,
                        )
                    )

        if (
            len(sample_by_source) == 3
            and sum(processing_state_paths.values()) > 0
        ):
            # Do not break: processing_state coverage must be corpus-wide.
            pass


for source_type, record in sorted(
    sample_by_source.items()
):
    print(f"\n--- {source_type.upper()} ---")
    print(
        "document_id:",
        record["document_id"],
    )

    for field in [
        "schema_version",
        "text_sha256",
        "raw_content_hash",
        "content_origin",
        "lineage",
        "source_metadata",
    ]:
        value = record.get(field)

        serialized = json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
        )

        if len(serialized) > 2200:
            serialized = (
                serialized[:2200]
                + "\n..."
            )

        print(
            f"\n{field}:"
        )
        print(serialized)


print("\n=== processing_state discovery ===")

if processing_state_paths:
    print(
        "processing_state locations found:"
    )

    for path, count in sorted(
        processing_state_paths.items()
    ):
        print(
            f"  {path}: {count}"
        )
else:
    print(
        "No processing_state field found "
        "inside final Common Document corpus."
    )


print("\n=== Contract conclusions ===")

print(
    "Common schema closed:",
    current_schema.get(
        "additionalProperties"
    ) is False,
)

print(
    "source_metadata allows undeclared fields:",
    (
        schema_properties["source_metadata"].get(
            "additionalProperties",
            "<not specified>",
        )
        is not False
    ),
)

print(
    "processing_state found in final corpus:",
    bool(processing_state_paths),
)

print(
    "\nNB08 REPORT + PROCESSING STATE "
    "CONTRACT DISCOVERY: COMPLETE"
)

=== NB07 report contract discovery ===

Report: reports/phase2/unified_schema_lineage_acceptance_v1.json
Top-level type: dict
Top-level keys: ['acceptance_status', 'acceptance_type', 'acceptance_version', 'artifacts', 'notebook', 'population', 'processing_boundaries', 'quality_gates', 'transcript_attribution', 'website_attribution']
  acceptance_status: "PASS"
  acceptance_type: "phase2_notebook_07_unified_schema_and_lineage"
  acceptance_version: "v1"
  artifacts: {"document_schema": {"exists": true, "path": "schemas/phase2/document.schema.json", "sha256": "3f577c71e8faa3b50c231bd4163cb5425158d0390e5fe5fcf39e00f530baa755", "size_bytes": 15673}, "lineage_report": {"exists": true, "path": "reports/phase2/unified_schema_lineage_report_v1.json", "sha256": "8e29e4817a1b0be22e3d877f37e0fa7b7b5af5dfe8cb1db4581a9c9be2487cad", "size_bytes": 12998}, "unified_documents": {"exists": true, "path": "data/phase2/processed/documents_unified_v1.jsonl", "sha256": "4ad6c3bcb1580d49f692e1d8a0406ee4c08bbf

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Prototype قرارداد خروجی NB08 در حافظه</h3>

<p>
Discovery نشان داد Common Document فعلی دارای <code>additionalProperties: false</code> است؛ بنابراین فیلدهایی مانند <code>raw_text</code>، <code>normalized_text</code> و <code>transformation_log</code> نباید مستقیماً به آن اضافه شوند.
</p>

<p>
همچنین فقط Transcriptها در <code>source_metadata</code> دارای <code>processing_state</code> هستند. بنابراین NB08 نباید ساختار جدیدی را به‌صورت اجباری به Book و Website تزریق کند.
</p>

<p>
قرارداد پیشنهادی در این مرحله دو Artifact منطقی دارد، اما هنوز هیچ فایلی نوشته نمی‌شود:
</p>

<ul>
<li><strong>Normalized Common Document Dataset:</strong> همان <code>common_document_v1</code> با حفظ کامل هویت و Lineage؛ فقط <code>text</code> نرمال می‌شود و <code>text_sha256</code> براساس متن جدید دوباره محاسبه می‌شود.</li>
<li><strong>Normalization Sidecar:</strong> برای هر Document شامل <code>raw_text</code>، <code>normalized_text</code>، Hash هر دو، نسخه Policy و Transformation Log است.</li>
</ul>

<p>
<code>raw_content_hash</code>، <code>document_id</code>، <code>source_id</code>، <code>lineage</code>، <code>attribution</code>، <code>rights</code> و سایر Metadataهای Common Document نباید تغییر کنند.
</p>

<h4>معیار PASS</h4>

<p>
Prototype برای تمام <code>19456</code> سند در حافظه ساخته می‌شود. تمام Common Documentهای نرمال‌شده باید Schema-valid باشند، خروجی Logged Normalizer باید دقیقاً با Normalizer اصلی برابر باشد و تنها دو Field مجاز به تغییر باشند: <code>text</code> و <code>text_sha256</code>.
</p>

</div>

In [14]:
# Code Cell 14

import copy
import hashlib
import json
import re
import unicodedata

from collections import Counter
from jsonschema import Draft202012Validator


common_document_validator = Draft202012Validator(
    current_schema
)

sidecar_contract_version = (
    "persian_normalization_record_v1"
)


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def normalize_persian_text_with_log(
    text,
    policy=normalization_policy,
):
    if not isinstance(text, str):
        raise TypeError(
            f"text must be str, got "
            f"{type(text).__name__}"
        )

    transformation_log = {
        "nfc": {
            "changed": False,
            "character_delta": 0,
        },
        "soft_hyphen": {
            "removed": 0,
        },
        "control_actions": {},
        "nbsp": {
            "occurrences": 0,
            "character_delta": 0,
        },
        "digit_mapping": {},
    }

    # Stage 1 — NFC
    nfc_text = unicodedata.normalize(
        policy["unicode_form"],
        text,
    )

    transformation_log["nfc"]["changed"] = (
        nfc_text != text
    )
    transformation_log["nfc"]["character_delta"] = (
        len(nfc_text) - len(text)
    )

    # Stage 2 — Soft hyphen
    soft_hyphen_rule = (
        policy["transformations"]["soft_hyphen"]
    )

    soft_hyphen_character = (
        soft_hyphen_rule["character"]
    )

    soft_hyphen_count = nfc_text.count(
        soft_hyphen_character
    )

    transformation_log["soft_hyphen"][
        "removed"
    ] = soft_hyphen_count

    if soft_hyphen_rule["action"] == "remove":
        control_input = nfc_text.replace(
            soft_hyphen_character,
            "",
        )
    else:
        raise ValueError(
            "Unsupported soft-hyphen action: "
            f"{soft_hyphen_rule['action']}"
        )

    # Stage 3 — Context-aware controls
    control_output = []
    control_action_counts = Counter()

    control_rules = (
        policy["transformations"]
        ["control_characters"]
    )

    for index, character in enumerate(
        control_input
    ):
        config = control_rules.get(character)

        if config is None:
            control_output.append(character)
            continue

        left = (
            control_input[index - 1]
            if index > 0
            else None
        )

        right = (
            control_input[index + 1]
            if index + 1 < len(control_input)
            else None
        )

        pattern = (
            classify_neighbor(left),
            classify_neighbor(right),
        )

        action = config["rules"].get(
            pattern,
            config["default_action"],
        )

        key = (
            f"{config['label']} | "
            f"{pattern[0]} -> {pattern[1]} | "
            f"{action}"
        )

        control_action_counts[key] += 1

        if action == "remove":
            continue

        if action == "replace_with_zwnj":
            control_output.append("\u200c")
            continue

        if action == "preserve":
            control_output.append(character)
            continue

        raise ValueError(
            f"Unknown control action: {action}"
        )

    control_text = "".join(control_output)

    transformation_log["control_actions"] = dict(
        sorted(control_action_counts.items())
    )

    # Stage 4 — Context-aware NBSP
    nbsp_rule = (
        policy["transformations"]
        ["contextual_rules"]
        ["nbsp_boundary"]
    )

    nbsp = nbsp_rule["source"]

    nbsp_occurrences = control_text.count(nbsp)

    transformation_log["nbsp"][
        "occurrences"
    ] = nbsp_occurrences

    if (
        nbsp_rule[
            "collapse_spacing_run_containing_nbsp"
        ]
    ):
        after_nbsp = re.sub(
            rf"[ {re.escape(nbsp)}]*"
            rf"{re.escape(nbsp)}"
            rf"[ {re.escape(nbsp)}]*",
            nbsp_rule["target"],
            control_text,
        )
    else:
        after_nbsp = control_text.replace(
            nbsp,
            nbsp_rule["target"],
        )

    transformation_log["nbsp"][
        "character_delta"
    ] = (
        len(after_nbsp)
        - len(control_text)
    )

    # Stage 5 — Arabic-Indic digits
    digit_text = after_nbsp

    digit_counts = Counter()

    for source, target in (
        policy["transformations"]
        ["digit_mapping"]
        .items()
    ):
        count = digit_text.count(source)

        if count:
            digit_counts[
                f"{source} -> {target}"
            ] += count

            digit_text = digit_text.replace(
                source,
                target,
            )

    transformation_log["digit_mapping"] = dict(
        sorted(digit_counts.items())
    )

    return digit_text, transformation_log


records_prototyped = 0
changed_documents = 0

normalized_schema_failures = []
canonical_mismatches = []
immutable_field_failures = []
sidecar_contract_failures = []

aggregate_control_actions = Counter()
aggregate_digit_actions = Counter()

aggregate_nfc_changed = 0
aggregate_soft_hyphen_removed = 0
aggregate_nbsp_occurrences = 0

sample_normalized_document = None
sample_sidecar_record = None

allowed_changed_fields = {
    "text",
    "text_sha256",
}


with dataset_path.open(
    "r",
    encoding="utf-8",
) as handle:
    for line in handle:
        if not line.strip():
            continue

        original_record = json.loads(line)
        raw_text = original_record["text"]

        (
            normalized_text,
            transformation_log,
        ) = normalize_persian_text_with_log(
            raw_text
        )

        canonical_normalized = (
            normalize_persian_text(raw_text)
        )

        records_prototyped += 1

        if normalized_text != canonical_normalized:
            canonical_mismatches.append(
                original_record["document_id"]
            )

        normalized_record = copy.deepcopy(
            original_record
        )

        normalized_record["text"] = (
            normalized_text
        )

        normalized_record["text_sha256"] = (
            sha256_text(normalized_text)
        )

        schema_errors = list(
            common_document_validator.iter_errors(
                normalized_record
            )
        )

        if schema_errors:
            normalized_schema_failures.append(
                {
                    "document_id": (
                        original_record["document_id"]
                    ),
                    "errors": [
                        error.message
                        for error in schema_errors[:5]
                    ],
                }
            )

        changed_field_names = {
            key
            for key in original_record
            if original_record.get(key)
            != normalized_record.get(key)
        }

        unexpected_changed_fields = (
            changed_field_names
            - allowed_changed_fields
        )

        if unexpected_changed_fields:
            immutable_field_failures.append(
                {
                    "document_id": (
                        original_record["document_id"]
                    ),
                    "fields": sorted(
                        unexpected_changed_fields
                    ),
                }
            )

        if normalized_text != raw_text:
            changed_documents += 1

        sidecar_record = {
            "contract_version": (
                sidecar_contract_version
            ),
            "policy_version": (
                normalization_policy[
                    "policy_version"
                ]
            ),
            "policy_revision": (
                normalization_policy[
                    "policy_revision"
                ]
            ),
            "document_id": (
                original_record["document_id"]
            ),
            "source_type": (
                original_record["source_type"]
            ),
            "source_id": (
                original_record["source_id"]
            ),
            "raw_text": raw_text,
            "raw_text_sha256": (
                original_record["text_sha256"]
            ),
            "normalized_text": normalized_text,
            "normalized_text_sha256": (
                normalized_record["text_sha256"]
            ),
            "text_changed": (
                raw_text != normalized_text
            ),
            "transformation_log": (
                transformation_log
            ),
        }

        required_sidecar_fields = {
            "contract_version",
            "policy_version",
            "policy_revision",
            "document_id",
            "source_type",
            "source_id",
            "raw_text",
            "raw_text_sha256",
            "normalized_text",
            "normalized_text_sha256",
            "text_changed",
            "transformation_log",
        }

        missing_sidecar_fields = (
            required_sidecar_fields
            - set(sidecar_record)
        )

        raw_hash_matches = (
            sidecar_record["raw_text_sha256"]
            == sha256_text(
                sidecar_record["raw_text"]
            )
        )

        normalized_hash_matches = (
            sidecar_record[
                "normalized_text_sha256"
            ]
            == sha256_text(
                sidecar_record[
                    "normalized_text"
                ]
            )
        )

        if (
            missing_sidecar_fields
            or not raw_hash_matches
            or not normalized_hash_matches
        ):
            sidecar_contract_failures.append(
                {
                    "document_id": (
                        original_record["document_id"]
                    ),
                    "missing_fields": sorted(
                        missing_sidecar_fields
                    ),
                    "raw_hash_matches": (
                        raw_hash_matches
                    ),
                    "normalized_hash_matches": (
                        normalized_hash_matches
                    ),
                }
            )

        if transformation_log["nfc"]["changed"]:
            aggregate_nfc_changed += 1

        aggregate_soft_hyphen_removed += (
            transformation_log[
                "soft_hyphen"
            ]["removed"]
        )

        aggregate_nbsp_occurrences += (
            transformation_log[
                "nbsp"
            ]["occurrences"]
        )

        aggregate_control_actions.update(
            transformation_log[
                "control_actions"
            ]
        )

        aggregate_digit_actions.update(
            transformation_log[
                "digit_mapping"
            ]
        )

        if (
            sample_normalized_document is None
            and normalized_text != raw_text
        ):
            sample_normalized_document = (
                normalized_record
            )
            sample_sidecar_record = (
                sidecar_record
            )


print("Records prototyped:", records_prototyped)
print("Changed documents:", changed_documents)

print("\nNormalized Common Document:")
print(
    "  Schema failures:",
    len(normalized_schema_failures),
)
print(
    "  Immutable-field failures:",
    len(immutable_field_failures),
)
print(
    "  Canonical normalizer mismatches:",
    len(canonical_mismatches),
)

print("\nSidecar contract:")
print(
    "  Contract version:",
    sidecar_contract_version,
)
print(
    "  Contract failures:",
    len(sidecar_contract_failures),
)

print("\nAggregate transformation evidence:")
print(
    "  NFC changed documents:",
    aggregate_nfc_changed,
)
print(
    "  Soft hyphens removed:",
    aggregate_soft_hyphen_removed,
)
print(
    "  NBSP occurrences:",
    aggregate_nbsp_occurrences,
)

print("\nControl actions:")
for label, count in sorted(
    aggregate_control_actions.items()
):
    print(
        f"  {label}: {count}"
    )

print("\nDigit actions:")
if aggregate_digit_actions:
    for label, count in sorted(
        aggregate_digit_actions.items()
    ):
        print(
            f"  {label}: {count}"
        )
else:
    print("  None")


print("\n=== Prototype changed Common Document ===")

assert sample_normalized_document is not None

print(
    json.dumps(
        {
            "document_id": (
                sample_normalized_document[
                    "document_id"
                ]
            ),
            "schema_version": (
                sample_normalized_document[
                    "schema_version"
                ]
            ),
            "text": (
                sample_normalized_document[
                    "text"
                ]
            ),
            "text_sha256": (
                sample_normalized_document[
                    "text_sha256"
                ]
            ),
            "raw_content_hash": (
                sample_normalized_document[
                    "raw_content_hash"
                ]
            ),
        },
        ensure_ascii=False,
        indent=2,
    )
)


print("\n=== Prototype sidecar record ===")

assert sample_sidecar_record is not None

preview_sidecar = copy.deepcopy(
    sample_sidecar_record
)

for field in [
    "raw_text",
    "normalized_text",
]:
    if len(preview_sidecar[field]) > 350:
        preview_sidecar[field] = (
            preview_sidecar[field][:350]
            + "..."
        )

print(
    json.dumps(
        preview_sidecar,
        ensure_ascii=False,
        indent=2,
    )
)


assert records_prototyped == record_count

assert changed_documents == 263, (
    f"Expected 263 changed documents, "
    f"found {changed_documents}"
)

assert not normalized_schema_failures, (
    f"Normalized schema failures: "
    f"{len(normalized_schema_failures)}"
)

assert not immutable_field_failures, (
    f"Unexpected immutable-field changes: "
    f"{len(immutable_field_failures)}"
)

assert not canonical_mismatches, (
    f"Logged/canonical mismatches: "
    f"{len(canonical_mismatches)}"
)

assert not sidecar_contract_failures, (
    f"Sidecar contract failures: "
    f"{len(sidecar_contract_failures)}"
)

assert aggregate_nfc_changed == 33

assert aggregate_nbsp_occurrences == 124

assert sum(
    aggregate_digit_actions.values()
) == 6

print(
    "\nNB08 IN-MEMORY OUTPUT "
    "CONTRACT PROTOTYPE: PASS"
)

Records prototyped: 19456
Changed documents: 263

Normalized Common Document:
  Schema failures: 0
  Immutable-field failures: 0
  Canonical normalizer mismatches: 0

Sidecar contract:
  Contract version: persian_normalization_record_v1
  Contract failures: 0

Aggregate transformation evidence:
  NFC changed documents: 33
  Soft hyphens removed: 0
  NBSP occurrences: 124

Control actions:
  BOM U+FEFF | LETTER -> LETTER | replace_with_zwnj: 205
  LRM U+200E | LETTER -> LETTER | replace_with_zwnj: 32
  LRM U+200E | LETTER -> PUNCT | remove: 1
  LRM U+200E | LETTER -> SPACE | remove: 3
  LRM U+200E | LETTER -> ZWNJ | remove: 25
  LRM U+200E | SPACE -> LETTER | remove: 2
  LRM U+200E | ZWNJ -> LETTER | remove: 9
  RLM U+200F | LETTER -> ZWNJ | remove: 7
  RLM U+200F | MARK -> SPACE | remove: 2
  RLM U+200F | ZWNJ -> LETTER | remove: 4
  Word joiner U+2060 | LETTER -> ZWNJ | remove: 3
  Word joiner U+2060 | ZWNJ -> LETTER | remove: 6
  ZWJ U+200D | ZWNJ -> LETTER | remove: 1

Digit actions

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Pre-write Gate پایدار و قابل اجرای مجدد</h3>

<p>
Pathهای خروجی NB08 براساس Convention واقعی Repository تثبیت شده‌اند.
این Gate باید هم در اجرای اولیه و هم پس از ساخته‌شدن Artifactها در
<code>Restart Kernel → Run All</code> قابل اجرا باشد.
</p>

<p>
بنابراین وجود یک Target File از این مرحله به بعد به‌خودی‌خود Failure نیست.
تطابق محتوای Artifact موجود با Candidate قطعی در Cell مربوط به Write بررسی خواهد شد؛
اگر فایل موجود با Candidate متفاوت باشد، Write متوقف می‌شود.
</p>

<h4>Artifactهای NB08</h4>

<ul>
<li><code>data/phase2/processed/documents_normalized_v1.jsonl</code></li>
<li><code>data/phase2/processed/persian_normalization_records_v1.jsonl</code></li>
<li><code>reports/phase2/persian_normalization_report_v1.json</code></li>
<li><code>reports/phase2/persian_normalization_acceptance_v1.json</code></li>
</ul>

<h4>قواعد</h4>

<ul>
<li>Input Dataset هرگز overwrite نمی‌شود.</li>
<li>Parent Directory همه Targetها باید موجود باشد.</li>
<li>Target اگر وجود دارد باید File باشد، نه Directory.</li>
<li>وجود Artifact قبلی فقط برای اجرای Idempotent مجاز است.</li>
<li>هیچ Write در این Cell انجام نمی‌شود.</li>
</ul>

</div>

In [15]:
# Code Cell 15

from pathlib import Path


normalized_dataset_path = (
    repo_root
    / "data"
    / "phase2"
    / "processed"
    / "documents_normalized_v1.jsonl"
)

normalization_sidecar_path = (
    repo_root
    / "data"
    / "phase2"
    / "processed"
    / "persian_normalization_records_v1.jsonl"
)

normalization_report_path = (
    repo_root
    / "reports"
    / "phase2"
    / "persian_normalization_report_v1.json"
)

normalization_acceptance_path = (
    repo_root
    / "reports"
    / "phase2"
    / "persian_normalization_acceptance_v1.json"
)


output_paths = {
    "normalized_dataset": normalized_dataset_path,
    "normalization_sidecar": normalization_sidecar_path,
    "normalization_report": normalization_report_path,
    "normalization_acceptance": normalization_acceptance_path,
}


print("=== NB08 stable output contract ===")

existing_files = {}
invalid_existing_targets = {}
missing_parents = {}


for label, path in output_paths.items():
    parent_exists = path.parent.is_dir()
    target_exists = path.exists()
    target_is_file = path.is_file()

    print(f"\n{label}:")
    print(
        "  path:",
        path.relative_to(repo_root),
    )
    print(
        "  parent exists:",
        parent_exists,
    )
    print(
        "  target exists:",
        target_exists,
    )

    if target_exists:
        print(
            "  target is file:",
            target_is_file,
        )

    if not parent_exists:
        missing_parents[label] = path.parent

    if target_exists and not target_is_file:
        invalid_existing_targets[label] = path

    if target_is_file:
        existing_files[label] = path


print("\n=== Safety checks ===")

print(
    "Input dataset:",
    dataset_path.relative_to(repo_root),
)

print(
    "Input/output collision:",
    dataset_path.resolve()
    == normalized_dataset_path.resolve(),
)

print(
    "Existing artifact files:",
    len(existing_files),
)

print(
    "Invalid existing targets:",
    len(invalid_existing_targets),
)

print(
    "Missing parents:",
    len(missing_parents),
)

print(
    "Checkpoint directory currently exists:",
    checkpoints_root.exists(),
)


if existing_files:
    print("\nExisting files allowed for idempotent validation:")

    for label, path in sorted(
        existing_files.items()
    ):
        print(
            f"  {label}: "
            f"{path.relative_to(repo_root)}"
        )


if invalid_existing_targets:
    print("\nInvalid targets:")

    for label, path in sorted(
        invalid_existing_targets.items()
    ):
        print(
            f"  {label}: "
            f"{path.relative_to(repo_root)}"
        )


if missing_parents:
    print("\nMissing parents:")

    for label, path in sorted(
        missing_parents.items()
    ):
        print(
            f"  {label}: "
            f"{path.relative_to(repo_root)}"
        )


assert (
    dataset_path.resolve()
    != normalized_dataset_path.resolve()
), (
    "Normalized output must not overwrite "
    "the unified input dataset."
)

assert not missing_parents, (
    "One or more output parent directories "
    "do not exist."
)

assert not invalid_existing_targets, (
    "One or more output targets exist "
    "but are not regular files."
)

assert normalized_dataset_path.suffix == ".jsonl"
assert normalization_sidecar_path.suffix == ".jsonl"
assert normalization_report_path.suffix == ".json"
assert normalization_acceptance_path.suffix == ".json"


print(
    "\nNB08 IDEMPOTENT PRE-WRITE ARTIFACT GATE: PASS"
)

=== NB08 stable output contract ===

normalized_dataset:
  path: data/phase2/processed/documents_normalized_v1.jsonl
  parent exists: True
  target exists: True
  target is file: True

normalization_sidecar:
  path: data/phase2/processed/persian_normalization_records_v1.jsonl
  parent exists: True
  target exists: True
  target is file: True

normalization_report:
  path: reports/phase2/persian_normalization_report_v1.json
  parent exists: True
  target exists: True
  target is file: True

normalization_acceptance:
  path: reports/phase2/persian_normalization_acceptance_v1.json
  parent exists: True
  target exists: True
  target is file: True

=== Safety checks ===
Input dataset: data/phase2/processed/documents_unified_v1.jsonl
Input/output collision: False
Existing artifact files: 4
Invalid existing targets: 0
Missing parents: 0
Checkpoint directory currently exists: False

Existing files allowed for idempotent validation:
  normalization_acceptance: reports/phase2/persian_normalizat

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Consistency Discovery — Metadata مرتبط با Normalization</h3>

<p>
پیش از ساخت Artifact نهایی، باید معنای Metadataهای موجود Transcript در ارتباط با Persian Normalization بررسی شود.
</p>

<p>
Transcriptها از مرحله Ingestion دارای
<code>source_metadata.processing_state.persian_normalization</code>
و
<code>source_metadata.text_state</code>
هستند.
در این مرحله هنوز فرض نمی‌کنیم که این Metadataها باید در NB08 تغییر کنند؛ ابتدا Coverage و مقادیر واقعی آن‌ها در کل Corpus کشف می‌شود.
</p>

<h4>هدف این Cell</h4>

<ul>
<li>کشف Corpus-wide تمام مقادیر <code>processing_state.persian_normalization</code>؛</li>
<li>کشف Corpus-wide تمام مقادیر <code>source_metadata.text_state</code>؛</li>
<li>بررسی توزیع این Fieldها براساس Source؛</li>
<li>فراهم‌کردن Evidence برای تشخیص اینکه این Metadataها وضعیت جاری Document هستند یا Provenance مرحله Ingestion.</li>
</ul>

<p>
تصمیم نهایی درباره حفظ یا تغییر این Metadataها در Discoveryهای بعدی و با مراجعه به قرارداد NB06/NB07 گرفته می‌شود.
این مرحله فقط Read Only است و هیچ Artifactی نوشته یا تغییر داده نمی‌شود.
</p>

</div>

In [16]:
# Code Cell 16

processing_state_presence = Counter()
persian_normalization_values = Counter()

text_state_presence = Counter()
text_state_values = Counter()

normalization_related_metadata = Counter()

records_checked = 0


with dataset_path.open(
    "r",
    encoding="utf-8",
) as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        source_type = record["source_type"]
        source_metadata = record["source_metadata"]

        records_checked += 1

        processing_state = source_metadata.get(
            "processing_state"
        )

        if isinstance(processing_state, dict):
            processing_state_presence[
                source_type
            ] += 1

            if "persian_normalization" in processing_state:
                value = processing_state[
                    "persian_normalization"
                ]

                persian_normalization_values[
                    (
                        source_type,
                        type(value).__name__,
                        repr(value),
                    )
                ] += 1

        if "text_state" in source_metadata:
            text_state_presence[
                source_type
            ] += 1

            value = source_metadata["text_state"]

            text_state_values[
                (
                    source_type,
                    type(value).__name__,
                    repr(value),
                )
            ] += 1

        # Discover all metadata keys whose name mentions normalization.
        stack = [
            (
                "$.source_metadata",
                source_metadata,
            )
        ]

        while stack:
            path, value = stack.pop()

            if isinstance(value, dict):
                for key, child in value.items():
                    child_path = f"{path}.{key}"

                    if (
                        "normaliz"
                        in key.lower()
                    ):
                        normalization_related_metadata[
                            (
                                source_type,
                                child_path,
                                repr(child),
                            )
                        ] += 1

                    stack.append(
                        (
                            child_path,
                            child,
                        )
                    )

            elif isinstance(value, list):
                for index, child in enumerate(value):
                    stack.append(
                        (
                            f"{path}[{index}]",
                            child,
                        )
                    )


print("Records checked:", records_checked)

print(
    "\n=== processing_state presence by source ==="
)

for source_type in sorted(source_type_counts):
    print(
        f"  {source_type}: "
        f"{processing_state_presence[source_type]}"
    )


print(
    "\n=== persian_normalization values ==="
)

for (
    source_type,
    value_type,
    value_repr,
), count in sorted(
    persian_normalization_values.items()
):
    print(
        f"  {source_type} | "
        f"{value_type} | "
        f"{value_repr}: {count}"
    )


print(
    "\n=== text_state presence by source ==="
)

for source_type in sorted(source_type_counts):
    print(
        f"  {source_type}: "
        f"{text_state_presence[source_type]}"
    )


print(
    "\n=== text_state values ==="
)

for (
    source_type,
    value_type,
    value_repr,
), count in sorted(
    text_state_values.items()
):
    print(
        f"  {source_type} | "
        f"{value_type} | "
        f"{value_repr}: {count}"
    )


print(
    "\n=== All normalization-related metadata ==="
)

if normalization_related_metadata:
    for (
        source_type,
        path,
        value_repr,
    ), count in sorted(
        normalization_related_metadata.items()
    ):
        print(
            f"  {source_type} | "
            f"{path} | "
            f"{value_repr}: {count}"
        )
else:
    print("  None")


assert records_checked == record_count

print(
    "\nNB08 NORMALIZATION METADATA "
    "CONSISTENCY DISCOVERY: COMPLETE"
)

Records checked: 19456

=== processing_state presence by source ===
  book: 0
  transcript: 16226
  website: 0

=== persian_normalization values ===
  transcript | bool | False: 16226

=== text_state presence by source ===
  book: 0
  transcript: 16226
  website: 0

=== text_state values ===
  transcript | str | 'SOURCE_PRESERVED_NOT_NORMALIZED': 16226

=== All normalization-related metadata ===
  transcript | $.source_metadata.processing_state.persian_normalization | False: 16226

NB08 NORMALIZATION METADATA CONSISTENCY DISCOVERY: COMPLETE


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Discovery مقدار معتبر Processing State پس از Normalization</h3>

<p>
برای Transcriptها دو Metadata موجود باید با Artifact نرمال‌شده سازگار شوند:
</p>

<ul>
<li><code>source_metadata.processing_state.persian_normalization</code></li>
<li><code>source_metadata.text_state</code></li>
</ul>

<p>
مقدار اول طبیعتاً Boolean است، اما مقدار معتبر <code>text_state</code> نباید حدس زده شود. در این مرحله Repository برای تمام استفاده‌ها و مقادیر واقعی این دو Field بررسی می‌شود.
</p>

<p>
هیچ فایل یا Datasetی در این Cell تغییر نمی‌کند.
</p>

</div>

In [17]:
# Code Cell 17

search_targets = [
    "text_state",
    "SOURCE_PRESERVED_NOT_NORMALIZED",
    "persian_normalization",
]

searchable_suffixes = {
    ".py",
    ".json",
    ".yaml",
    ".yml",
    ".md",
    ".ipynb",
    ".toml",
}

excluded_parts = {
    ".git",
    ".venv",
    "venv",
    "node_modules",
    "__pycache__",
}

max_file_size = 4 * 1024 * 1024
max_hits_per_target = 50

hits = {
    target: []
    for target in search_targets
}


for path in repo_root.rglob("*"):
    if not path.is_file():
        continue

    relative = path.relative_to(repo_root)

    if any(
        part in excluded_parts
        for part in relative.parts
    ):
        continue

    if "data" in relative.parts:
        continue

    if path.suffix.lower() not in searchable_suffixes:
        continue

    if path.stat().st_size > max_file_size:
        continue

    try:
        content = path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    except Exception:
        continue

    lowered = content.lower()

    for target in search_targets:
        target_lower = target.lower()
        search_from = 0

        while (
            len(hits[target])
            < max_hits_per_target
        ):
            index = lowered.find(
                target_lower,
                search_from,
            )

            if index == -1:
                break

            line_number = (
                content.count(
                    "\n",
                    0,
                    index,
                )
                + 1
            )

            line_start = (
                content.rfind(
                    "\n",
                    0,
                    index,
                )
                + 1
            )

            line_end = content.find(
                "\n",
                index,
            )

            if line_end == -1:
                line_end = len(content)

            excerpt = content[
                line_start:line_end
            ].strip()

            if len(excerpt) > 300:
                excerpt = (
                    excerpt[:300]
                    + "..."
                )

            hits[target].append(
                {
                    "path": str(relative),
                    "line": line_number,
                    "excerpt": excerpt,
                }
            )

            search_from = (
                index + len(target_lower)
            )


print(
    "=== Repository processing-state convention discovery ==="
)

for target in search_targets:
    print(
        f"\nTarget: {target} "
        f"| hits={len(hits[target])}"
    )

    if not hits[target]:
        print("  None")
        continue

    for hit in hits[target]:
        print(
            f"  {hit['path']}:{hit['line']}"
        )
        print(
            f"    {hit['excerpt']}"
        )


print(
    "\n=== Explicit candidate state strings discovered ==="
)

candidate_state_strings = Counter()

for hit in hits["text_state"]:
    excerpt = hit["excerpt"]

    for token in re.findall(
        r'["\']([A-Z][A-Z0-9_]{3,})["\']',
        excerpt,
    ):
        candidate_state_strings[token] += 1

if candidate_state_strings:
    for state, count in sorted(
        candidate_state_strings.items()
    ):
        print(
            f"  {state}: {count}"
        )
else:
    print("  None")


print(
    "\nNB08 PROCESSING STATE VALUE DISCOVERY: COMPLETE"
)

=== Repository processing-state convention discovery ===

Target: text_state | hits=50
  schemas/phase2/document.schema.json:394
    "text_state"
  notebooks/phase2/08_persian_normalization.ipynb:4659
    "  \"text_state\": \"SOURCE_PRESERVED_NOT_NORMALIZED\",\n",
  notebooks/phase2/08_persian_normalization.ipynb:6050
    "<code>source_metadata.text_state</code>\n",
  notebooks/phase2/08_persian_normalization.ipynb:6059
    "<li>کشف Corpus-wide تمام مقادیر <code>source_metadata.text_state</code>؛</li>\n",
  notebooks/phase2/08_persian_normalization.ipynb:6092
    "=== text_state presence by source ===\n",
  notebooks/phase2/08_persian_normalization.ipynb:6097
    "=== text_state values ===\n",
  notebooks/phase2/08_persian_normalization.ipynb:6113
    "text_state_presence = Counter()\n",
  notebooks/phase2/08_persian_normalization.ipynb:6114
    "text_state_values = Counter()\n",
  notebooks/phase2/08_persian_normalization.ipynb:6157
    "        if \"text_state\" in source_metadata:\n

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Semantic Discovery — معنای واقعی text_state و processing_state</h3>

<p>
Repository هیچ مقدار استاندارد دیگری برای <code>text_state</code> تعریف نکرده است. بنابراین پیش از تصمیم درباره تغییر یا حفظ این Metadata، باید Context اصلی ایجاد و مصرف آن در NB06 و NB07 بررسی شود.
</p>

<p>
این مرحله Source Cellهای مرتبط را مستقیماً از Notebookهای قبلی استخراج می‌کند تا مشخص شود:
</p>

<ul>
<li><code>text_state</code> وضعیت متن در مرحله Transcript Ingestion است یا وضعیت فعلی Document؛</li>
<li><code>processing_state.persian_normalization</code> برای Provenance مرحله قبلی نگهداری می‌شود یا باید با هر پردازش بعدی Update شود؛</li>
<li>NB07 هنگام ساخت Common Document این Metadataها را با چه معنایی Preserve کرده است.</li>
</ul>

<p>
این مرحله کاملاً Read Only است و هیچ Artifactی تغییر نمی‌کند.
</p>

</div>

In [18]:
# Code Cell 18

import json
from pathlib import Path


notebooks_to_inspect = [
    repo_root
    / "notebooks"
    / "phase2"
    / "06_transcript_ingestion.ipynb",

    repo_root
    / "notebooks"
    / "phase2"
    / "07_unified_schema_and_lineage.ipynb",
]

search_terms = [
    "text_state",
    "SOURCE_PRESERVED_NOT_NORMALIZED",
    "persian_normalization",
    "processing_state",
]

max_matching_cells_per_notebook = 20


print(
    "=== Semantic context discovery from NB06 / NB07 ==="
)

for notebook_path in notebooks_to_inspect:
    assert notebook_path.exists(), (
        f"Notebook missing: {notebook_path}"
    )

    notebook = json.loads(
        notebook_path.read_text(
            encoding="utf-8"
        )
    )

    print(
        f"\n\n######## "
        f"{notebook_path.relative_to(repo_root)} "
        f"########"
    )

    matching_cells = []

    for cell_index, cell in enumerate(
        notebook.get("cells", [])
    ):
        source = "".join(
            cell.get("source", [])
        )

        matched_terms = [
            term
            for term in search_terms
            if term.lower() in source.lower()
        ]

        if not matched_terms:
            continue

        matching_cells.append(
            {
                "cell_index": cell_index,
                "cell_type": cell.get(
                    "cell_type"
                ),
                "matched_terms": (
                    matched_terms
                ),
                "source": source,
            }
        )

        if (
            len(matching_cells)
            >= max_matching_cells_per_notebook
        ):
            break

    print(
        "Matching cells shown:",
        len(matching_cells),
    )

    for item in matching_cells:
        print(
            "\n--- Cell index:",
            item["cell_index"],
            "| type:",
            item["cell_type"],
            "| matches:",
            item["matched_terms"],
            "---",
        )

        source = item["source"]

        if len(source) > 4000:
            source = (
                source[:4000]
                + "\n...[TRUNCATED]..."
            )

        print(source)


print(
    "\nNB08 PROCESSING METADATA "
    "SEMANTIC DISCOVERY: COMPLETE"
)

=== Semantic context discovery from NB06 / NB07 ===


######## notebooks/phase2/06_transcript_ingestion.ipynb ########
Matching cells shown: 6

--- Cell index: 36 | type: code | matches: ['persian_normalization'] ---
# Code Cell 18

explicit_bold_marker_programs = {
    "program:asre-shirin",
    "program:khane-to",
    "program:uni-tehran",
}

chehelston_contextual_path_set = set(
    chehelston_contextual_paths
)

manifest_aliases_by_program = {
    record["program_id"]: dict(record.get("speaker_aliases", {}))
    for record in program_records
}

parser_contract_v1 = {
    "contract_version": "transcript_parser_contract_v1",
    "timestamp_policy": "PRESERVE_REAL_ELSE_NONE",
    "missing_timestamp_value": None,
    "unknown_speaker_inference_allowed": False,
    "preserve_raw_text": True,
    "persian_normalization_allowed": False,
    "deduplication_allowed": False,
    "chunking_allowed": False,
    "policies": {
        "HUMAN_CONFIRMED_SOLO": {
            "strategy": "SOLO_MANIF

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت و ثبت Artifactهای اصلی Persian Normalization</h3>

<p>
پس از PASS شدن تمام Gateهای Discovery، Policy، Dry Run، Transformation Audit و Notebook Integrity، اکنون دو Artifact اصلی NB08 ساخته می‌شوند.
</p>

<ul>
<li><code>documents_normalized_v1.jsonl</code>: نسخه نرمال‌شده Common Documentها؛ فقط <code>text</code> و <code>text_sha256</code> براساس Policy NB08 تغییر می‌کنند.</li>
<li><code>persian_normalization_records_v1.jsonl</code>: Sidecar کامل شامل Raw Text، Normalized Text، Hashها، Policy و Transformation Log برای تمام 19,456 Document.</li>
</ul>

<p>
Metadataهای Source-specific از جمله وضعیت Transcript Ingestion بدون تغییر حفظ می‌شوند، زیرا Discoveryهای قبلی نشان دادند این اطلاعات Provenance مرحله NB06/NB07 هستند.
</p>

<h4>قواعد Write</h4>

<ul>
<li>Candidateها ابتدا در فایل Temporary ساخته و Validate می‌شوند.</li>
<li>اگر Artifact نهایی وجود نداشته باشد، Candidate به‌صورت Atomic ثبت می‌شود.</li>
<li>اگر Artifact از قبل وجود داشته باشد، فقط در صورت تطابق Byte-for-byte / SHA-256 دوباره استفاده می‌شود.</li>
<li>Artifact موجود با محتوای متفاوت هرگز overwrite نمی‌شود.</li>
<li>Input Dataset دست‌نخورده باقی می‌ماند.</li>
</ul>

</div>

In [19]:
# Code Cell 19

import copy
import hashlib
import json
import os
import tempfile
from collections import Counter
from pathlib import Path

from jsonschema import Draft202012Validator


# ---------------------------------------------------------
# Independent schema validator
# ---------------------------------------------------------

normalized_schema_path = (
    repo_root
    / "schemas"
    / "phase2"
    / "document.schema.json"
)

assert normalized_schema_path.is_file(), (
    f"Schema not found: {normalized_schema_path}"
)

with normalized_schema_path.open(
    "r",
    encoding="utf-8",
) as handle:
    normalized_schema = json.load(handle)

Draft202012Validator.check_schema(
    normalized_schema
)

normalized_validator = (
    Draft202012Validator(
        normalized_schema
    )
)


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def sha256_file_stream(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def deterministic_json_line(record):
    return (
        json.dumps(
            record,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        + "\n"
    )


def install_idempotent_candidate(
    candidate_path,
    final_path,
):
    candidate_hash = sha256_file_stream(
        candidate_path
    )

    candidate_size = (
        candidate_path.stat().st_size
    )

    if final_path.exists():
        assert final_path.is_file(), (
            "Existing target is not a file: "
            f"{final_path}"
        )

        existing_hash = sha256_file_stream(
            final_path
        )

        existing_size = (
            final_path.stat().st_size
        )

        if (
            existing_hash != candidate_hash
            or existing_size != candidate_size
        ):
            candidate_path.unlink(
                missing_ok=True
            )

            raise AssertionError(
                "Existing artifact differs from "
                "deterministic NB08 candidate: "
                f"{final_path.relative_to(repo_root)}"
            )

        candidate_path.unlink(
            missing_ok=True
        )

        return {
            "action": "REUSED_IDENTICAL",
            "sha256": existing_hash,
            "size_bytes": existing_size,
        }

    os.replace(
        candidate_path,
        final_path,
    )

    return {
        "action": "CREATED",
        "sha256": candidate_hash,
        "size_bytes": candidate_size,
    }


# ---------------------------------------------------------
# Temporary candidates
# ---------------------------------------------------------

normalized_temp = None
sidecar_temp = None

records_written = 0
changed_documents = 0

source_counts = Counter()
changed_by_source = Counter()

schema_failures = []
sidecar_failures = []
canonical_mismatches = []

raw_hash_failures = []
normalized_hash_failures = []

immutable_field_failures = []

transformation_totals = {
    "nfc_changed_documents": 0,
    "soft_hyphens_removed": 0,
    "nbsp_occurrences": 0,
    "control_actions": Counter(),
    "digit_mapping": Counter(),
}


try:
    normalized_fd, normalized_temp_name = (
        tempfile.mkstemp(
            prefix="nb08_normalized_",
            suffix=".jsonl.tmp",
            dir=normalized_dataset_path.parent,
        )
    )

    os.close(normalized_fd)

    normalized_temp = Path(
        normalized_temp_name
    )


    sidecar_fd, sidecar_temp_name = (
        tempfile.mkstemp(
            prefix="nb08_sidecar_",
            suffix=".jsonl.tmp",
            dir=normalization_sidecar_path.parent,
        )
    )

    os.close(sidecar_fd)

    sidecar_temp = Path(
        sidecar_temp_name
    )


    with (
        dataset_path.open(
            "r",
            encoding="utf-8",
        ) as source_handle,
        normalized_temp.open(
            "w",
            encoding="utf-8",
            newline="\n",
        ) as normalized_handle,
        sidecar_temp.open(
            "w",
            encoding="utf-8",
            newline="\n",
        ) as sidecar_handle,
    ):

        for line_number, line in enumerate(
            source_handle,
            start=1,
        ):
            if not line.strip():
                continue

            original_record = json.loads(
                line
            )

            raw_text = original_record[
                "text"
            ]

            normalized_text, transformation_log = (
                normalize_persian_text_with_log(
                    raw_text,
                    normalization_policy,
                )
            )

            canonical_text = (
                normalize_persian_text(
                    raw_text,
                    normalization_policy,
                )
            )

            if normalized_text != canonical_text:
                canonical_mismatches.append(
                    original_record[
                        "document_id"
                    ]
                )


            normalized_record = copy.deepcopy(
                original_record
            )

            normalized_record[
                "text"
            ] = normalized_text

            normalized_record[
                "text_sha256"
            ] = hashlib.sha256(
                normalized_text.encode(
                    "utf-8"
                )
            ).hexdigest()


            # ---------------------------------------------
            # Common Document schema validation
            # ---------------------------------------------

            schema_errors = list(
                normalized_validator.iter_errors(
                    normalized_record
                )
            )

            if schema_errors:
                schema_failures.append(
                    {
                        "document_id": (
                            original_record[
                                "document_id"
                            ]
                        ),
                        "error_count": len(
                            schema_errors
                        ),
                    }
                )


            # ---------------------------------------------
            # Hash validation
            # ---------------------------------------------

            raw_text_hash = hashlib.sha256(
                raw_text.encode("utf-8")
            ).hexdigest()

            if (
                raw_text_hash
                != original_record[
                    "text_sha256"
                ]
            ):
                raw_hash_failures.append(
                    original_record[
                        "document_id"
                    ]
                )


            normalized_text_hash = (
                hashlib.sha256(
                    normalized_text.encode(
                        "utf-8"
                    )
                ).hexdigest()
            )

            if (
                normalized_text_hash
                != normalized_record[
                    "text_sha256"
                ]
            ):
                normalized_hash_failures.append(
                    original_record[
                        "document_id"
                    ]
                )


            # ---------------------------------------------
            # Immutable-field validation
            # ---------------------------------------------

            allowed_changed_fields = {
                "text",
                "text_sha256",
            }

            changed_fields = {
                key
                for key in original_record
                if (
                    original_record[key]
                    != normalized_record[key]
                )
            }

            unexpected_changes = (
                changed_fields
                - allowed_changed_fields
            )

            if unexpected_changes:
                immutable_field_failures.append(
                    {
                        "document_id": (
                            original_record[
                                "document_id"
                            ]
                        ),
                        "fields": sorted(
                            unexpected_changes
                        ),
                    }
                )


            # ---------------------------------------------
            # Sidecar record
            # ---------------------------------------------

            sidecar_record = {
                "contract_version": (
                    "persian_normalization_record_v1"
                ),
                "policy_version": (
                    normalization_policy[
                        "policy_version"
                    ]
                ),
                "policy_revision": (
                    normalization_policy[
                        "policy_revision"
                    ]
                ),
                "document_id": (
                    original_record[
                        "document_id"
                    ]
                ),
                "source_type": (
                    original_record[
                        "source_type"
                    ]
                ),
                "source_id": (
                    original_record[
                        "source_id"
                    ]
                ),
                "raw_text": raw_text,
                "raw_text_sha256": (
                    raw_text_hash
                ),
                "normalized_text": (
                    normalized_text
                ),
                "normalized_text_sha256": (
                    normalized_text_hash
                ),
                "text_changed": (
                    raw_text
                    != normalized_text
                ),
                "transformation_log": (
                    transformation_log
                ),
            }


            required_sidecar_fields = {
                "contract_version",
                "policy_version",
                "policy_revision",
                "document_id",
                "source_type",
                "source_id",
                "raw_text",
                "raw_text_sha256",
                "normalized_text",
                "normalized_text_sha256",
                "text_changed",
                "transformation_log",
            }

            if (
                set(sidecar_record)
                != required_sidecar_fields
            ):
                sidecar_failures.append(
                    original_record[
                        "document_id"
                    ]
                )


            # ---------------------------------------------
            # Aggregate evidence
            # ---------------------------------------------

            source_type = (
                original_record[
                    "source_type"
                ]
            )

            source_counts[
                source_type
            ] += 1


            if raw_text != normalized_text:
                changed_documents += 1

                changed_by_source[
                    source_type
                ] += 1


            if transformation_log[
                "nfc"
            ][
                "changed"
            ]:
                transformation_totals[
                    "nfc_changed_documents"
                ] += 1


            transformation_totals[
                "soft_hyphens_removed"
            ] += transformation_log[
                "soft_hyphen"
            ][
                "removed"
            ]


            transformation_totals[
                "nbsp_occurrences"
            ] += transformation_log[
                "nbsp"
            ][
                "occurrences"
            ]


            for (
                action_key,
                action_count,
            ) in transformation_log[
                "control_actions"
            ].items():
                transformation_totals[
                    "control_actions"
                ][
                    action_key
                ] += action_count


            for (
                digit_key,
                digit_count,
            ) in transformation_log[
                "digit_mapping"
            ].items():
                transformation_totals[
                    "digit_mapping"
                ][
                    digit_key
                ] += digit_count


            # ---------------------------------------------
            # Persist deterministic candidate JSONL
            # ---------------------------------------------

            normalized_handle.write(
                deterministic_json_line(
                    normalized_record
                )
            )

            sidecar_handle.write(
                deterministic_json_line(
                    sidecar_record
                )
            )

            records_written += 1


    # -----------------------------------------------------
    # Pre-install gates
    # -----------------------------------------------------

    assert (
        records_written
        == record_count
        == 19456
    )

    assert changed_documents == 263

    assert dict(source_counts) == {
        "website": 529,
        "book": 2701,
        "transcript": 16226,
    }

    assert dict(changed_by_source) == {
        "website": 118,
        "book": 132,
        "transcript": 13,
    }

    assert not schema_failures, (
        f"Schema failures: "
        f"{schema_failures[:10]}"
    )

    assert not sidecar_failures, (
        f"Sidecar failures: "
        f"{sidecar_failures[:10]}"
    )

    assert not canonical_mismatches, (
        "Logged normalizer differs from "
        "canonical normalizer."
    )

    assert not raw_hash_failures, (
        f"Raw text hash failures: "
        f"{raw_hash_failures[:10]}"
    )

    assert not normalized_hash_failures, (
        f"Normalized hash failures: "
        f"{normalized_hash_failures[:10]}"
    )

    assert not immutable_field_failures, (
        "Unexpected Common Document "
        "field modifications detected."
    )

    assert (
        transformation_totals[
            "nfc_changed_documents"
        ]
        == 33
    )

    assert (
        transformation_totals[
            "soft_hyphens_removed"
        ]
        == 0
    )

    assert (
        transformation_totals[
            "nbsp_occurrences"
        ]
        == 124
    )

    assert (
        sum(
            transformation_totals[
                "digit_mapping"
            ].values()
        )
        == 6
    )


    # -----------------------------------------------------
    # Install / reuse final artifacts
    # -----------------------------------------------------

    normalized_state = (
        install_idempotent_candidate(
            normalized_temp,
            normalized_dataset_path,
        )
    )

    normalized_temp = None


    sidecar_state = (
        install_idempotent_candidate(
            sidecar_temp,
            normalization_sidecar_path,
        )
    )

    sidecar_temp = None


finally:
    if (
        normalized_temp is not None
        and normalized_temp.exists()
    ):
        normalized_temp.unlink()

    if (
        sidecar_temp is not None
        and sidecar_temp.exists()
    ):
        sidecar_temp.unlink()


# ---------------------------------------------------------
# Final report
# ---------------------------------------------------------

print("NB08 CORE ARTIFACT WRITE")
print("=" * 90)

print("\nPOPULATION")
print(
    "  Records written:",
    records_written,
)
print(
    "  Changed documents:",
    changed_documents,
)
print(
    "  Changed by source:",
    dict(changed_by_source),
)

print("\nNORMALIZED DATASET")
print(
    "  Path:",
    normalized_dataset_path.relative_to(
        repo_root
    ),
)
print(
    "  Action:",
    normalized_state["action"],
)
print(
    "  Size bytes:",
    normalized_state["size_bytes"],
)
print(
    "  SHA-256:",
    normalized_state["sha256"],
)

print("\nNORMALIZATION SIDECAR")
print(
    "  Path:",
    normalization_sidecar_path.relative_to(
        repo_root
    ),
)
print(
    "  Action:",
    sidecar_state["action"],
)
print(
    "  Size bytes:",
    sidecar_state["size_bytes"],
)
print(
    "  SHA-256:",
    sidecar_state["sha256"],
)

print("\nVALIDATION")
print(
    "  Schema failures:",
    len(schema_failures),
)
print(
    "  Immutable-field failures:",
    len(
        immutable_field_failures
    ),
)
print(
    "  Canonical mismatches:",
    len(canonical_mismatches),
)
print(
    "  Raw hash failures:",
    len(raw_hash_failures),
)
print(
    "  Normalized hash failures:",
    len(normalized_hash_failures),
)
print(
    "  Sidecar failures:",
    len(sidecar_failures),
)

print(
    "\nNB08 CORE NORMALIZATION ARTIFACT WRITE: PASS"
)

NB08 CORE ARTIFACT WRITE

POPULATION
  Records written: 19456
  Changed documents: 263
  Changed by source: {'website': 118, 'book': 132, 'transcript': 13}

NORMALIZED DATASET
  Path: data/phase2/processed/documents_normalized_v1.jsonl
  Action: REUSED_IDENTICAL
  Size bytes: 63804043
  SHA-256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40

NORMALIZATION SIDECAR
  Path: data/phase2/processed/persian_normalization_records_v1.jsonl
  Action: REUSED_IDENTICAL
  Size bytes: 25084621
  SHA-256: ea73c8e9b1547c11c58a0f5b845a0ca8e5932a6d72392ff81feb44f9404e6ef7

VALIDATION
  Schema failures: 0
  Immutable-field failures: 0
  Canonical mismatches: 0
  Raw hash failures: 0
  Normalized hash failures: 0
  Sidecar failures: 0

NB08 CORE NORMALIZATION ARTIFACT WRITE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Reload مستقل از Disk و اعتبارسنجی Artifactهای اصلی NB08</h3>

<p>
دو Artifact اصلی Persian Normalization اکنون از روی Disk و مستقل از Candidateهای زمان Write دوباره خوانده و اعتبارسنجی می‌شوند.
</p>

<p>
Dataset ورودی، Dataset نرمال‌شده و Sidecar به‌صورت رکوردبه‌رکورد تطبیق داده می‌شوند تا صحت Schema، Hash، Lineage، Raw/Normalized Text، Transformation Log و محدودبودن تغییرات به Policy مصوب NB08 اثبات شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هر سه Stream دقیقاً 19,456 رکورد داشته باشند.</li>
<li>Document ID تکراری وجود نداشته باشد.</li>
<li>تمام Common Documentهای نرمال‌شده Schema-valid باشند.</li>
<li>فقط <code>text</code> و <code>text_sha256</code> نسبت به Dataset ورودی تغییر کرده باشند.</li>
<li>Sidecar با هر دو نسخه Raw و Normalized تطابق کامل داشته باشد.</li>
<li>Normalized Text دوباره با Normalizer رسمی NB08 تطابق داشته باشد.</li>
<li>هیچ حذف کاراکتر توضیح‌داده‌نشده‌ای خارج از Transformation Log وجود نداشته باشد.</li>
<li>Residualهای الزامی مانند BOM، NBSP، LRM/RLM، Word Joiner، ZWJ و Arabic-Indic Digits صفر باشند.</li>
</ul>

</div>

In [20]:
# Code Cell 20

import hashlib
import itertools
import json
import unicodedata
from collections import Counter

from jsonschema import Draft202012Validator


# ---------------------------------------------------------
# Independent schema reload
# ---------------------------------------------------------

disk_schema_path = (
    repo_root
    / "schemas"
    / "phase2"
    / "document.schema.json"
)

assert disk_schema_path.is_file()

with disk_schema_path.open(
    "r",
    encoding="utf-8",
) as handle:
    disk_schema = json.load(handle)

Draft202012Validator.check_schema(
    disk_schema
)

disk_validator = Draft202012Validator(
    disk_schema
)


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def independent_file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def jsonl_stream(path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                raise AssertionError(
                    f"Blank JSONL line: "
                    f"{path.name}:{line_number}"
                )

            yield (
                line_number,
                json.loads(line),
            )


def expected_character_delta_from_log(
    transformation_log,
):
    expected_delta = 0

    expected_delta += (
        transformation_log[
            "nfc"
        ][
            "character_delta"
        ]
    )

    expected_delta -= (
        transformation_log[
            "soft_hyphen"
        ][
            "removed"
        ]
    )

    expected_delta += (
        transformation_log[
            "nbsp"
        ][
            "character_delta"
        ]
    )

    for action_key, count in (
        transformation_log[
            "control_actions"
        ].items()
    ):
        if str(action_key).endswith(
            "| remove"
        ):
            expected_delta -= count

        elif str(action_key).endswith(
            "| replace_with_zwnj"
        ):
            expected_delta += 0

        else:
            raise AssertionError(
                "Unknown control action in "
                f"transformation log: {action_key}"
            )

    # Digit replacements are 1-to-1.
    return expected_delta


# ---------------------------------------------------------
# Artifact existence + hashes
# ---------------------------------------------------------

required_disk_artifacts = {
    "input_dataset": dataset_path,
    "normalized_dataset": normalized_dataset_path,
    "normalization_sidecar": normalization_sidecar_path,
    "document_schema": disk_schema_path,
}

missing_disk_artifacts = [
    name
    for name, path
    in required_disk_artifacts.items()
    if not path.is_file()
]

assert not missing_disk_artifacts, (
    f"Missing artifacts: "
    f"{missing_disk_artifacts}"
)

disk_artifact_hashes = {
    name: independent_file_sha256(
        path
    )
    for name, path
    in required_disk_artifacts.items()
}


# ---------------------------------------------------------
# Independent record-by-record validation
# ---------------------------------------------------------

records_validated = 0

document_ids = set()
duplicate_document_ids = []

source_distribution = Counter()
changed_by_source_disk = Counter()

schema_failures_disk = []
text_hash_failures_disk = []
canonical_failures_disk = []
immutable_failures_disk = []

sidecar_contract_failures = []
sidecar_alignment_failures = []
sidecar_hash_failures = []

unexplained_character_deltas = []

raw_character_count = 0
normalized_character_count = 0

non_nfc_documents = []

residual_counts = Counter()

arabic_yeh_after = 0
arabic_kaf_after = 0
zwnj_after = 0

ascii_digits_after = 0
persian_digits_after = 0

required_sidecar_fields = {
    "contract_version",
    "policy_version",
    "policy_revision",
    "document_id",
    "source_type",
    "source_id",
    "raw_text",
    "raw_text_sha256",
    "normalized_text",
    "normalized_text_sha256",
    "text_changed",
    "transformation_log",
}


input_stream = jsonl_stream(
    dataset_path
)

normalized_stream = jsonl_stream(
    normalized_dataset_path
)

sidecar_stream = jsonl_stream(
    normalization_sidecar_path
)


for triplet in itertools.zip_longest(
    input_stream,
    normalized_stream,
    sidecar_stream,
    fillvalue=None,
):
    input_item, normalized_item, sidecar_item = (
        triplet
    )

    assert input_item is not None, (
        "Normalized/Sidecar stream contains "
        "more records than input."
    )

    assert normalized_item is not None, (
        "Normalized dataset ended early."
    )

    assert sidecar_item is not None, (
        "Normalization sidecar ended early."
    )


    input_line, original_record = input_item
    normalized_line, normalized_record = (
        normalized_item
    )
    sidecar_line, sidecar_record = (
        sidecar_item
    )


    assert (
        input_line
        == normalized_line
        == sidecar_line
    )


    document_id = original_record[
        "document_id"
    ]

    source_type = original_record[
        "source_type"
    ]


    # -----------------------------------------------------
    # Identity / population
    # -----------------------------------------------------

    if document_id in document_ids:
        duplicate_document_ids.append(
            document_id
        )

    document_ids.add(
        document_id
    )

    source_distribution[
        source_type
    ] += 1


    # -----------------------------------------------------
    # Alignment
    # -----------------------------------------------------

    if (
        normalized_record[
            "document_id"
        ]
        != document_id
        or sidecar_record[
            "document_id"
        ]
        != document_id
    ):
        sidecar_alignment_failures.append(
            document_id
        )


    # -----------------------------------------------------
    # Common Document schema
    # -----------------------------------------------------

    schema_errors = list(
        disk_validator.iter_errors(
            normalized_record
        )
    )

    if schema_errors:
        schema_failures_disk.append(
            {
                "document_id": document_id,
                "error_count": len(
                    schema_errors
                ),
            }
        )


    # -----------------------------------------------------
    # Text hashes
    # -----------------------------------------------------

    computed_normalized_hash = (
        hashlib.sha256(
            normalized_record[
                "text"
            ].encode("utf-8")
        ).hexdigest()
    )

    if (
        computed_normalized_hash
        != normalized_record[
            "text_sha256"
        ]
    ):
        text_hash_failures_disk.append(
            document_id
        )


    # -----------------------------------------------------
    # Canonical normalization from raw input
    # -----------------------------------------------------

    canonical_normalized_text = (
        normalize_persian_text(
            original_record[
                "text"
            ],
            normalization_policy,
        )
    )

    if (
        canonical_normalized_text
        != normalized_record[
            "text"
        ]
    ):
        canonical_failures_disk.append(
            document_id
        )


    # -----------------------------------------------------
    # Immutable fields
    # -----------------------------------------------------

    changed_fields = {
        key
        for key in original_record
        if (
            original_record[key]
            != normalized_record[key]
        )
    }

    unexpected_fields = (
        changed_fields
        - {
            "text",
            "text_sha256",
        }
    )

    if unexpected_fields:
        immutable_failures_disk.append(
            {
                "document_id": document_id,
                "fields": sorted(
                    unexpected_fields
                ),
            }
        )


    # -----------------------------------------------------
    # Sidecar contract
    # -----------------------------------------------------

    if (
        set(sidecar_record)
        != required_sidecar_fields
    ):
        sidecar_contract_failures.append(
            document_id
        )


    raw_text = original_record[
        "text"
    ]

    normalized_text = normalized_record[
        "text"
    ]


    computed_raw_hash = hashlib.sha256(
        raw_text.encode("utf-8")
    ).hexdigest()

    if not (
        sidecar_record[
            "contract_version"
        ]
        == "persian_normalization_record_v1"

        and sidecar_record[
            "policy_version"
        ]
        == normalization_policy[
            "policy_version"
        ]

        and sidecar_record[
            "policy_revision"
        ]
        == normalization_policy[
            "policy_revision"
        ]

        and sidecar_record[
            "source_type"
        ]
        == source_type

        and sidecar_record[
            "source_id"
        ]
        == original_record[
            "source_id"
        ]

        and sidecar_record[
            "raw_text"
        ]
        == raw_text

        and sidecar_record[
            "normalized_text"
        ]
        == normalized_text

        and sidecar_record[
            "text_changed"
        ]
        == (
            raw_text
            != normalized_text
        )
    ):
        sidecar_alignment_failures.append(
            document_id
        )


    if not (
        sidecar_record[
            "raw_text_sha256"
        ]
        == computed_raw_hash
        == original_record[
            "text_sha256"
        ]
    ):
        sidecar_hash_failures.append(
            document_id
        )


    if not (
        sidecar_record[
            "normalized_text_sha256"
        ]
        == computed_normalized_hash
        == normalized_record[
            "text_sha256"
        ]
    ):
        sidecar_hash_failures.append(
            document_id
        )


    # -----------------------------------------------------
    # Explain every character deletion
    # -----------------------------------------------------

    actual_delta = (
        len(normalized_text)
        - len(raw_text)
    )

    expected_delta = (
        expected_character_delta_from_log(
            sidecar_record[
                "transformation_log"
            ]
        )
    )

    if actual_delta != expected_delta:
        unexplained_character_deltas.append(
            {
                "document_id": document_id,
                "actual_delta": actual_delta,
                "expected_delta": expected_delta,
            }
        )


    raw_character_count += len(
        raw_text
    )

    normalized_character_count += len(
        normalized_text
    )


    # -----------------------------------------------------
    # Changed distribution
    # -----------------------------------------------------

    if raw_text != normalized_text:
        changed_by_source_disk[
            source_type
        ] += 1


    # -----------------------------------------------------
    # Residual normalization audit
    # -----------------------------------------------------

    if (
        unicodedata.normalize(
            "NFC",
            normalized_text,
        )
        != normalized_text
    ):
        non_nfc_documents.append(
            document_id
        )


    residual_characters = {
        "BOM_U+FEFF": "\ufeff",
        "SOFT_HYPHEN_U+00AD": "\u00ad",
        "ZWJ_U+200D": "\u200d",
        "LRM_U+200E": "\u200e",
        "RLM_U+200F": "\u200f",
        "WORD_JOINER_U+2060": "\u2060",
        "NBSP_U+00A0": "\u00a0",
    }

    for label, character in (
        residual_characters.items()
    ):
        residual_counts[
            label
        ] += normalized_text.count(
            character
        )


    for digit in "٠١٢٣٤٥٦٧٨٩":
        residual_counts[
            "ARABIC_INDIC_DIGITS"
        ] += normalized_text.count(
            digit
        )


    arabic_yeh_after += (
        normalized_text.count(
            "\u064a"
        )
    )

    arabic_kaf_after += (
        normalized_text.count(
            "\u0643"
        )
    )

    zwnj_after += (
        normalized_text.count(
            "\u200c"
        )
    )

    ascii_digits_after += sum(
        character.isascii()
        and character.isdigit()
        for character in normalized_text
    )

    persian_digits_after += sum(
        character in "۰۱۲۳۴۵۶۷۸۹"
        for character in normalized_text
    )


    records_validated += 1


# ---------------------------------------------------------
# Acceptance gates
# ---------------------------------------------------------

assert records_validated == 19456

assert len(document_ids) == 19456
assert not duplicate_document_ids

assert dict(source_distribution) == {
    "website": 529,
    "book": 2701,
    "transcript": 16226,
}

assert dict(changed_by_source_disk) == {
    "website": 118,
    "book": 132,
    "transcript": 13,
}

assert sum(
    changed_by_source_disk.values()
) == 263

assert not schema_failures_disk
assert not text_hash_failures_disk
assert not canonical_failures_disk
assert not immutable_failures_disk

assert not sidecar_contract_failures
assert not sidecar_alignment_failures
assert not sidecar_hash_failures

assert not unexplained_character_deltas

assert raw_character_count == 3125877
assert normalized_character_count == 3125699

assert (
    normalized_character_count
    - raw_character_count
    == -178
)

assert not non_nfc_documents

assert all(
    count == 0
    for count in residual_counts.values()
)

assert arabic_yeh_after == 137
assert arabic_kaf_after == 45
assert zwnj_after == 59493

assert ascii_digits_after == 6883


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print("NB08 INDEPENDENT DISK VALIDATION")
print("=" * 90)

print("\nARTIFACT HASHES")

for name, digest in (
    disk_artifact_hashes.items()
):
    print(
        f"  {name}: {digest}"
    )


print("\nPOPULATION")
print(
    "  Records validated:",
    records_validated,
)
print(
    "  Unique document IDs:",
    len(document_ids),
)
print(
    "  Source distribution:",
    dict(source_distribution),
)
print(
    "  Changed by source:",
    dict(changed_by_source_disk),
)


print("\nCOMMON DOCUMENT")
print(
    "  Schema failures:",
    len(schema_failures_disk),
)
print(
    "  Text hash failures:",
    len(text_hash_failures_disk),
)
print(
    "  Canonical normalization failures:",
    len(canonical_failures_disk),
)
print(
    "  Immutable-field failures:",
    len(immutable_failures_disk),
)


print("\nSIDECAR")
print(
    "  Contract failures:",
    len(sidecar_contract_failures),
)
print(
    "  Alignment failures:",
    len(sidecar_alignment_failures),
)
print(
    "  Hash failures:",
    len(sidecar_hash_failures),
)


print("\nCHARACTER SAFETY")
print(
    "  Raw characters:",
    raw_character_count,
)
print(
    "  Normalized characters:",
    normalized_character_count,
)
print(
    "  Character delta:",
    (
        normalized_character_count
        - raw_character_count
    ),
)
print(
    "  Unexplained character deltas:",
    len(
        unexplained_character_deltas
    ),
)
print(
    "  Non-NFC documents:",
    len(non_nfc_documents),
)


print("\nRESIDUAL REQUIRED CLEANUP")
print(
    " ",
    dict(residual_counts),
)


print("\nPRESERVED / EXPECTED")
print(
    "  Arabic yeh U+064A:",
    arabic_yeh_after,
)
print(
    "  Arabic kaf U+0643:",
    arabic_kaf_after,
)
print(
    "  ZWNJ U+200C:",
    zwnj_after,
)
print(
    "  ASCII digits:",
    ascii_digits_after,
)
print(
    "  Persian digits:",
    persian_digits_after,
)


print(
    "\nNB08 INDEPENDENT DISK ARTIFACT "
    "VALIDATION: PASS"
)

NB08 INDEPENDENT DISK VALIDATION

ARTIFACT HASHES
  input_dataset: 4ad6c3bcb1580d49f692e1d8a0406ee4c08bbf580cbb473e37baa107e89f5f04
  normalized_dataset: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
  normalization_sidecar: ea73c8e9b1547c11c58a0f5b845a0ca8e5932a6d72392ff81feb44f9404e6ef7
  document_schema: 3f577c71e8faa3b50c231bd4163cb5425158d0390e5fe5fcf39e00f530baa755

POPULATION
  Records validated: 19456
  Unique document IDs: 19456
  Source distribution: {'website': 529, 'book': 2701, 'transcript': 16226}
  Changed by source: {'website': 118, 'book': 132, 'transcript': 13}

COMMON DOCUMENT
  Schema failures: 0
  Text hash failures: 0
  Canonical normalization failures: 0
  Immutable-field failures: 0

SIDECAR
  Contract failures: 0
  Alignment failures: 0
  Hash failures: 0

CHARACTER SAFETY
  Raw characters: 3125877
  Normalized characters: 3125699
  Character delta: -178
  Unexplained character deltas: 0
  Non-NFC documents: 0

RESIDUAL REQUIRED CLEANUP
  {'B

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت گزارش فنی Persian Normalization</h3>

<p>
پس از اعتبارسنجی مستقل Artifactهای اصلی، گزارش فنی نسخه‌دار NB08 از روی فایل‌های Persistشده ساخته می‌شود.
</p>

<p>
این گزارش شامل Policy، ورودی و خروجی‌ها، Hashها، جمعیت Corpus، آمار قبل و بعد، Transformation Evidence، تصمیم Preserve/Defer برای ی و ک عربی، Character Safety و نتیجه Validation مستقل است.
</p>

<p>
گزارش به‌صورت Deterministic و Idempotent در
<code>reports/phase2/persian_normalization_report_v1.json</code>
ثبت می‌شود.
</p>

</div>

In [21]:
# Code Cell 21

import hashlib
import json
from collections import Counter


# ---------------------------------------------------------
# Independent aggregation from persisted sidecar
# ---------------------------------------------------------

report_records = 0
report_changed_documents = 0

report_source_distribution = Counter()
report_changed_by_source = Counter()

report_nfc_changed_documents = 0
report_soft_hyphen_removed = 0
report_nbsp_occurrences = 0

report_control_actions = Counter()
report_digit_actions = Counter()

report_raw_characters = 0
report_normalized_characters = 0


with normalization_sidecar_path.open(
    "r",
    encoding="utf-8",
) as handle:

    for line_number, line in enumerate(
        handle,
        start=1,
    ):
        if not line.strip():
            raise AssertionError(
                f"Blank sidecar line: "
                f"{line_number}"
            )

        record = json.loads(line)

        report_records += 1

        source_type = record[
            "source_type"
        ]

        report_source_distribution[
            source_type
        ] += 1

        raw_text = record[
            "raw_text"
        ]

        normalized_text = record[
            "normalized_text"
        ]

        report_raw_characters += len(
            raw_text
        )

        report_normalized_characters += len(
            normalized_text
        )

        if record[
            "text_changed"
        ]:
            report_changed_documents += 1

            report_changed_by_source[
                source_type
            ] += 1


        transformation_log = record[
            "transformation_log"
        ]

        if transformation_log[
            "nfc"
        ][
            "changed"
        ]:
            report_nfc_changed_documents += 1


        report_soft_hyphen_removed += (
            transformation_log[
                "soft_hyphen"
            ][
                "removed"
            ]
        )


        report_nbsp_occurrences += (
            transformation_log[
                "nbsp"
            ][
                "occurrences"
            ]
        )


        for (
            action_key,
            action_count,
        ) in transformation_log[
            "control_actions"
        ].items():

            report_control_actions[
                action_key
            ] += action_count


        for (
            digit_key,
            digit_count,
        ) in transformation_log[
            "digit_mapping"
        ].items():

            report_digit_actions[
                digit_key
            ] += digit_count


# ---------------------------------------------------------
# Independent residual profile from normalized dataset
# ---------------------------------------------------------

report_residual_counts = Counter()

report_arabic_yeh = 0
report_arabic_kaf = 0
report_zwnj = 0

report_ascii_digits = 0
report_persian_digits = 0

report_non_nfc_documents = 0


with normalized_dataset_path.open(
    "r",
    encoding="utf-8",
) as handle:

    for line_number, line in enumerate(
        handle,
        start=1,
    ):
        if not line.strip():
            raise AssertionError(
                f"Blank normalized line: "
                f"{line_number}"
            )

        record = json.loads(line)

        text = record["text"]

        if (
            unicodedata.normalize(
                "NFC",
                text,
            )
            != text
        ):
            report_non_nfc_documents += 1


        residual_map = {
            "bom_u_feFF": "\ufeff",
            "soft_hyphen_u_00AD": "\u00ad",
            "zwj_u_200D": "\u200d",
            "lrm_u_200E": "\u200e",
            "rlm_u_200F": "\u200f",
            "word_joiner_u_2060": "\u2060",
            "nbsp_u_00A0": "\u00a0",
        }

        for label, character in (
            residual_map.items()
        ):
            report_residual_counts[
                label
            ] += text.count(
                character
            )


        report_residual_counts[
            "arabic_indic_digits"
        ] += sum(
            character in "٠١٢٣٤٥٦٧٨٩"
            for character in text
        )


        report_arabic_yeh += text.count(
            "\u064a"
        )

        report_arabic_kaf += text.count(
            "\u0643"
        )

        report_zwnj += text.count(
            "\u200c"
        )

        report_ascii_digits += sum(
            character.isascii()
            and character.isdigit()
            for character in text
        )

        report_persian_digits += sum(
            character in "۰۱۲۳۴۵۶۷۸۹"
            for character in text
        )


# ---------------------------------------------------------
# Report-level gates
# ---------------------------------------------------------

assert report_records == 19456

assert report_changed_documents == 263

assert dict(
    report_source_distribution
) == {
    "website": 529,
    "book": 2701,
    "transcript": 16226,
}

assert dict(
    report_changed_by_source
) == {
    "website": 118,
    "book": 132,
    "transcript": 13,
}

assert report_nfc_changed_documents == 33
assert report_soft_hyphen_removed == 0
assert report_nbsp_occurrences == 124

assert sum(
    report_digit_actions.values()
) == 6

assert report_raw_characters == 3125877
assert report_normalized_characters == 3125699

assert (
    report_normalized_characters
    - report_raw_characters
    == -178
)

assert report_non_nfc_documents == 0

assert all(
    value == 0
    for value in report_residual_counts.values()
)

assert report_arabic_yeh == 137
assert report_arabic_kaf == 45
assert report_zwnj == 59493

assert report_ascii_digits == 6883
assert report_persian_digits == 1600


# ---------------------------------------------------------
# JSON-safe policy evidence
# ---------------------------------------------------------

policy_control_rules = []

for (
    character,
    control_policy,
) in normalization_policy[
    "transformations"
][
    "control_characters"
].items():

    for (
        neighbor_pair,
        action,
    ) in control_policy[
        "rules"
    ].items():

        policy_control_rules.append(
            {
                "character_codepoint": (
                    f"U+{ord(character):04X}"
                ),
                "label": (
                    control_policy[
                        "label"
                    ]
                ),
                "left_context": (
                    neighbor_pair[0]
                ),
                "right_context": (
                    neighbor_pair[1]
                ),
                "action": action,
            }
        )


policy_control_rules = sorted(
    policy_control_rules,
    key=lambda item: (
        item[
            "character_codepoint"
        ],
        item[
            "left_context"
        ],
        item[
            "right_context"
        ],
        item[
            "action"
        ],
    ),
)


# ---------------------------------------------------------
# Build deterministic technical report
# ---------------------------------------------------------

normalization_report_v1 = {
    "report_type": (
        "phase2_persian_normalization_report"
    ),
    "report_version": "v1",
    "notebook": (
        "notebooks/phase2/"
        "08_persian_normalization.ipynb"
    ),

    "policy": {
        "policy_version": (
            normalization_policy[
                "policy_version"
            ]
        ),
        "policy_revision": (
            normalization_policy[
                "policy_revision"
            ]
        ),
        "unicode_form": "NFC",

        "scope": [
            "Common Document text field",
        ],

        "preserve": {
            "zwnj_u200c": True,
            "arabic_yeh_u064a": True,
            "arabic_kaf_u0643": True,
            "ascii_digits": True,
            "persian_digits": True,
            "newlines": True,
            "punctuation": True,
            "unknown_control_contexts": True,
        },

        "deferred": {
            "arabic_yeh_global_conversion": (
                "DEFERRED_NO_SAFE_CONTEXT_RULE"
            ),
            "arabic_kaf_global_conversion": (
                "DEFERRED_NO_SAFE_CONTEXT_RULE"
            ),
        },

        "control_rules": (
            policy_control_rules
        ),

        "arabic_indic_digit_mapping": {
            "٠": "۰",
            "١": "۱",
            "٢": "۲",
            "٣": "۳",
            "٤": "۴",
            "٥": "۵",
            "٦": "۶",
            "٧": "۷",
            "٨": "۸",
            "٩": "۹",
        },

        "forbidden_operations": (
            normalization_policy[
                "forbidden_operations"
            ]
        ),
    },

    "artifacts": {
        "input_dataset": {
            "path": (
                dataset_path.relative_to(
                    repo_root
                ).as_posix()
            ),
            "sha256": (
                independent_file_sha256(
                    dataset_path
                )
            ),
            "size_bytes": (
                dataset_path.stat().st_size
            ),
        },

        "normalized_dataset": {
            "path": (
                normalized_dataset_path
                .relative_to(
                    repo_root
                )
                .as_posix()
            ),
            "sha256": (
                independent_file_sha256(
                    normalized_dataset_path
                )
            ),
            "size_bytes": (
                normalized_dataset_path
                .stat()
                .st_size
            ),
        },

        "normalization_sidecar": {
            "path": (
                normalization_sidecar_path
                .relative_to(
                    repo_root
                )
                .as_posix()
            ),
            "sha256": (
                independent_file_sha256(
                    normalization_sidecar_path
                )
            ),
            "size_bytes": (
                normalization_sidecar_path
                .stat()
                .st_size
            ),
        },

        "document_schema": {
            "path": (
                disk_schema_path
                .relative_to(
                    repo_root
                )
                .as_posix()
            ),
            "sha256": (
                independent_file_sha256(
                    disk_schema_path
                )
            ),
            "size_bytes": (
                disk_schema_path
                .stat()
                .st_size
            ),
        },
    },

    "population": {
        "records": report_records,
        "source_distribution": dict(
            sorted(
                report_source_distribution.items()
            )
        ),
        "changed_documents": (
            report_changed_documents
        ),
        "unchanged_documents": (
            report_records
            - report_changed_documents
        ),
        "changed_by_source": dict(
            sorted(
                report_changed_by_source.items()
            )
        ),
    },

    "character_statistics": {
        "raw_characters": (
            report_raw_characters
        ),
        "normalized_characters": (
            report_normalized_characters
        ),
        "character_delta": (
            report_normalized_characters
            - report_raw_characters
        ),
        "nfc_changed_documents": (
            report_nfc_changed_documents
        ),
        "soft_hyphens_removed": (
            report_soft_hyphen_removed
        ),
        "nbsp_occurrences_normalized": (
            report_nbsp_occurrences
        ),
    },

    "transformation_evidence": {
        "control_actions": dict(
            sorted(
                report_control_actions.items()
            )
        ),
        "digit_mapping": dict(
            sorted(
                report_digit_actions.items()
            )
        ),
    },

    "post_normalization_profile": {
        "non_nfc_documents": (
            report_non_nfc_documents
        ),
        "residual_required_cleanup": dict(
            sorted(
                report_residual_counts.items()
            )
        ),
        "preserved_arabic_yeh_u064a": (
            report_arabic_yeh
        ),
        "preserved_arabic_kaf_u0643": (
            report_arabic_kaf
        ),
        "zwnj_u200c": report_zwnj,
        "ascii_digits": (
            report_ascii_digits
        ),
        "persian_digits": (
            report_persian_digits
        ),
    },

    "metadata_semantics": {
        "transcript_ingestion_metadata": (
            "PRESERVED_AS_SOURCE_PROVENANCE"
        ),
        "processing_state_persian_normalization": (
            "NOT_REWRITTEN_BY_NB08"
        ),
        "text_state": (
            "NOT_REWRITTEN_BY_NB08"
        ),
        "reason": (
            "NB06/NB07 discovery established "
            "these fields as source-stage "
            "provenance rather than NB08 "
            "current-processing state."
        ),
    },

    "validation": {
        "common_document_schema_failures": 0,
        "text_hash_failures": 0,
        "canonical_normalization_failures": 0,
        "immutable_field_failures": 0,
        "sidecar_contract_failures": 0,
        "sidecar_alignment_failures": 0,
        "sidecar_hash_failures": 0,
        "unexplained_character_deltas": 0,
        "duplicate_document_ids": 0,
        "status": "PASS",
    },
}


candidate_report_text = (
    json.dumps(
        normalization_report_v1,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )
    + "\n"
)


# ---------------------------------------------------------
# Idempotent report write
# ---------------------------------------------------------

if normalization_report_path.exists():
    existing_report_text = (
        normalization_report_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        existing_report_text
        == candidate_report_text
    ), (
        "Existing normalization report "
        "differs from deterministic candidate."
    )

    report_write_action = (
        "REUSED_IDENTICAL"
    )

else:
    normalization_report_path.write_text(
        candidate_report_text,
        encoding="utf-8",
    )

    report_write_action = "CREATED"


report_sha256 = (
    independent_file_sha256(
        normalization_report_path
    )
)

report_size_bytes = (
    normalization_report_path
    .stat()
    .st_size
)


# ---------------------------------------------------------
# Independent reload
# ---------------------------------------------------------

reloaded_report = json.loads(
    normalization_report_path.read_text(
        encoding="utf-8"
    )
)

assert (
    reloaded_report[
        "validation"
    ][
        "status"
    ]
    == "PASS"
)

assert (
    reloaded_report[
        "population"
    ][
        "records"
    ]
    == 19456
)

assert (
    reloaded_report[
        "population"
    ][
        "changed_documents"
    ]
    == 263
)

assert (
    reloaded_report[
        "artifacts"
    ][
        "normalized_dataset"
    ][
        "sha256"
    ]
    == independent_file_sha256(
        normalized_dataset_path
    )
)

assert (
    reloaded_report[
        "artifacts"
    ][
        "normalization_sidecar"
    ][
        "sha256"
    ]
    == independent_file_sha256(
        normalization_sidecar_path
    )
)


print("NB08 TECHNICAL REPORT")
print("=" * 90)

print(
    "Path:",
    normalization_report_path.relative_to(
        repo_root
    ),
)

print(
    "Action:",
    report_write_action,
)

print(
    "Size bytes:",
    report_size_bytes,
)

print(
    "SHA-256:",
    report_sha256,
)

print(
    "Records:",
    reloaded_report[
        "population"
    ][
        "records"
    ],
)

print(
    "Changed documents:",
    reloaded_report[
        "population"
    ][
        "changed_documents"
    ],
)

print(
    "Character delta:",
    reloaded_report[
        "character_statistics"
    ][
        "character_delta"
    ],
)

print(
    "Validation status:",
    reloaded_report[
        "validation"
    ][
        "status"
    ],
)

print(
    "\nNB08 PERSIAN NORMALIZATION "
    "TECHNICAL REPORT: PASS"
)

NB08 TECHNICAL REPORT
Path: reports/phase2/persian_normalization_report_v1.json
Action: REUSED_IDENTICAL
Size bytes: 7402
SHA-256: bd3ffd58bb97dd460970aa9bd9043bf85690214d4614557dc384de611dd761ca
Records: 19456
Changed documents: 263
Character delta: -178
Validation status: PASS

NB08 PERSIAN NORMALIZATION TECHNICAL REPORT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Acceptance نهایی Persian Normalization</h3>

<p>
پس از ساخت Artifactهای اصلی، Reload مستقل از Disk و تولید گزارش فنی، اکنون Acceptance رسمی Notebook 08 ساخته می‌شود.
</p>

<p>
این Artifact وضعیت نهایی Corpus، Hash فایل‌های پذیرفته‌شده، Quality Gateها، مرز پردازش و تصمیم‌های Policy را ثبت می‌کند.
</p>

<p>
Acceptance فقط زمانی PASS می‌شود که تمام Artifactهای اصلی موجود باشند، Hashهای آن‌ها با گزارش فنی تطابق داشته باشند و تمام Quality Gateهای NB08 بدون Failure باشند.
</p>

</div>

In [22]:
# Code Cell 22

import hashlib
import json


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def acceptance_file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ---------------------------------------------------------
# Required persisted artifacts
# ---------------------------------------------------------

accepted_artifact_paths = {
    "input_dataset": dataset_path,
    "document_schema": disk_schema_path,
    "normalized_dataset": normalized_dataset_path,
    "normalization_sidecar": normalization_sidecar_path,
    "technical_report": normalization_report_path,
}

missing_acceptance_artifacts = [
    name
    for name, path in accepted_artifact_paths.items()
    if not path.is_file()
]

assert not missing_acceptance_artifacts, (
    "Missing required NB08 artifacts: "
    f"{missing_acceptance_artifacts}"
)


accepted_artifacts = {}

for name, path in accepted_artifact_paths.items():
    accepted_artifacts[name] = {
        "path": (
            path.relative_to(
                repo_root
            ).as_posix()
        ),
        "sha256": (
            acceptance_file_sha256(
                path
            )
        ),
        "size_bytes": (
            path.stat().st_size
        ),
    }


# ---------------------------------------------------------
# Independent technical-report reload
# ---------------------------------------------------------

technical_report_disk = json.loads(
    normalization_report_path.read_text(
        encoding="utf-8"
    )
)

assert (
    technical_report_disk[
        "validation"
    ][
        "status"
    ]
    == "PASS"
)

assert (
    technical_report_disk[
        "population"
    ][
        "records"
    ]
    == 19456
)

assert (
    technical_report_disk[
        "population"
    ][
        "changed_documents"
    ]
    == 263
)

assert (
    technical_report_disk[
        "population"
    ][
        "unchanged_documents"
    ]
    == 19193
)


# ---------------------------------------------------------
# Cross-artifact hash reconciliation
# ---------------------------------------------------------

hash_reconciliation_failures = []

report_artifact_map = {
    "input_dataset": "input_dataset",
    "document_schema": "document_schema",
    "normalized_dataset": "normalized_dataset",
    "normalization_sidecar": "normalization_sidecar",
}

for (
    acceptance_name,
    report_name,
) in report_artifact_map.items():

    report_hash = (
        technical_report_disk[
            "artifacts"
        ][
            report_name
        ][
            "sha256"
        ]
    )

    actual_hash = (
        accepted_artifacts[
            acceptance_name
        ][
            "sha256"
        ]
    )

    if report_hash != actual_hash:
        hash_reconciliation_failures.append(
            {
                "artifact": acceptance_name,
                "report_sha256": report_hash,
                "actual_sha256": actual_hash,
            }
        )


assert not hash_reconciliation_failures, (
    "Technical-report artifact hashes "
    "do not match persisted files."
)


# ---------------------------------------------------------
# Quality gates
# ---------------------------------------------------------

quality_gates = {
    "record_count_19456": (
        technical_report_disk[
            "population"
        ][
            "records"
        ]
        == 19456
    ),

    "changed_document_count_263": (
        technical_report_disk[
            "population"
        ][
            "changed_documents"
        ]
        == 263
    ),

    "source_distribution_preserved": (
        technical_report_disk[
            "population"
        ][
            "source_distribution"
        ]
        == {
            "book": 2701,
            "transcript": 16226,
            "website": 529,
        }
    ),

    "changed_by_source_verified": (
        technical_report_disk[
            "population"
        ][
            "changed_by_source"
        ]
        == {
            "book": 132,
            "transcript": 13,
            "website": 118,
        }
    ),

    "common_document_schema_failures_zero": (
        technical_report_disk[
            "validation"
        ][
            "common_document_schema_failures"
        ]
        == 0
    ),

    "text_hash_failures_zero": (
        technical_report_disk[
            "validation"
        ][
            "text_hash_failures"
        ]
        == 0
    ),

    "canonical_normalization_failures_zero": (
        technical_report_disk[
            "validation"
        ][
            "canonical_normalization_failures"
        ]
        == 0
    ),

    "immutable_field_failures_zero": (
        technical_report_disk[
            "validation"
        ][
            "immutable_field_failures"
        ]
        == 0
    ),

    "sidecar_contract_failures_zero": (
        technical_report_disk[
            "validation"
        ][
            "sidecar_contract_failures"
        ]
        == 0
    ),

    "sidecar_alignment_failures_zero": (
        technical_report_disk[
            "validation"
        ][
            "sidecar_alignment_failures"
        ]
        == 0
    ),

    "sidecar_hash_failures_zero": (
        technical_report_disk[
            "validation"
        ][
            "sidecar_hash_failures"
        ]
        == 0
    ),

    "duplicate_document_ids_zero": (
        technical_report_disk[
            "validation"
        ][
            "duplicate_document_ids"
        ]
        == 0
    ),

    "unexplained_character_deltas_zero": (
        technical_report_disk[
            "validation"
        ][
            "unexplained_character_deltas"
        ]
        == 0
    ),

    "non_nfc_documents_zero": (
        technical_report_disk[
            "post_normalization_profile"
        ][
            "non_nfc_documents"
        ]
        == 0
    ),

    "required_cleanup_residuals_zero": (
        all(
            value == 0
            for value in (
                technical_report_disk[
                    "post_normalization_profile"
                ][
                    "residual_required_cleanup"
                ].values()
            )
        )
    ),

    "character_delta_verified": (
        technical_report_disk[
            "character_statistics"
        ][
            "character_delta"
        ]
        == -178
    ),

    "arabic_yeh_preserved": (
        technical_report_disk[
            "post_normalization_profile"
        ][
            "preserved_arabic_yeh_u064a"
        ]
        == 137
    ),

    "arabic_kaf_preserved": (
        technical_report_disk[
            "post_normalization_profile"
        ][
            "preserved_arabic_kaf_u0643"
        ]
        == 45
    ),

    "ascii_digits_preserved": (
        technical_report_disk[
            "post_normalization_profile"
        ][
            "ascii_digits"
        ]
        == 6883
    ),

    "technical_report_status_pass": (
        technical_report_disk[
            "validation"
        ][
            "status"
        ]
        == "PASS"
    ),
}


failed_quality_gates = [
    name
    for name, passed in quality_gates.items()
    if not passed
]

assert not failed_quality_gates, (
    "NB08 quality gates failed: "
    f"{failed_quality_gates}"
)


# ---------------------------------------------------------
# Corrected acceptance artifact
# ---------------------------------------------------------

normalization_acceptance_v1 = {
    "acceptance_type": (
        "phase2_persian_normalization_acceptance"
    ),
    "acceptance_version": "v1",

    "notebook": (
        "notebooks/phase2/"
        "08_persian_normalization.ipynb"
    ),

    "acceptance_status": "PASS",

    "policy": {
        "policy_version": (
            technical_report_disk[
                "policy"
            ][
                "policy_version"
            ]
        ),
        "policy_revision": (
            technical_report_disk[
                "policy"
            ][
                "policy_revision"
            ]
        ),
        "unicode_form": "NFC",
    },

    "artifacts": accepted_artifacts,

    "population": {
        "records": 19456,
        "source_distribution": {
            "website": 529,
            "book": 2701,
            "transcript": 16226,
        },
        "changed_documents": 263,
        "unchanged_documents": 19193,
        "changed_by_source": {
            "website": 118,
            "book": 132,
            "transcript": 13,
        },
    },

    "quality_gates": {
        key: "PASS"
        for key in sorted(
            quality_gates
        )
    },

    "processing_boundaries": {
        "persian_normalization": (
            "PERFORMED_IN_NB08"
        ),
        "raw_text_preservation": (
            "PRESERVED_IN_NORMALIZATION_SIDECAR"
        ),
        "text_sha256": (
            "RECOMPUTED_FOR_NORMALIZED_TEXT"
        ),
        "raw_content_hash": (
            "PRESERVED_UNCHANGED"
        ),
        "lineage": (
            "PRESERVED_UNCHANGED"
        ),
        "attribution": (
            "PRESERVED_UNCHANGED"
        ),
        "rights": (
            "PRESERVED_UNCHANGED"
        ),
        "source_metadata": (
            "PRESERVED_AS_SOURCE_PROVENANCE"
        ),
        "arabic_yeh_global_conversion": (
            "DEFERRED"
        ),
        "arabic_kaf_global_conversion": (
            "DEFERRED"
        ),
        "deduplication": (
            "NOT_PERFORMED_IN_NB08"
        ),
        "chunking": (
            "NOT_PERFORMED_IN_NB08"
        ),
        "embedding": (
            "NOT_PERFORMED_IN_NB08"
        ),
        "indexing": (
            "NOT_PERFORMED_IN_NB08"
        ),
    },

    "character_safety": {
        "raw_characters": 3125877,
        "normalized_characters": 3125699,
        "character_delta": -178,
        "unexplained_character_deltas": 0,
        "required_cleanup_residuals": 0,
    },

    "decision": {
        "status": "PASS",
        "next_notebook_allowed": True,
        "scope_boundary": (
            "NB08_COMPLETE_NB09_NOT_STARTED"
        ),
    },
}


candidate_acceptance_text = (
    json.dumps(
        normalization_acceptance_v1,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )
    + "\n"
)


# ---------------------------------------------------------
# Safe correction / idempotent write
# ---------------------------------------------------------

known_flawed_acceptance_sha256 = (
    "389c9d840103b227a103c469be419911"
    "c2c4be83f4ebbda34aa83cf4d7bb47a4"
)


if normalization_acceptance_path.exists():

    existing_acceptance_text = (
        normalization_acceptance_path
        .read_text(
            encoding="utf-8"
        )
    )

    existing_acceptance_hash = (
        acceptance_file_sha256(
            normalization_acceptance_path
        )
    )

    if (
        existing_acceptance_text
        == candidate_acceptance_text
    ):
        acceptance_write_action = (
            "REUSED_IDENTICAL"
        )

    else:
        existing_acceptance_json = json.loads(
            existing_acceptance_text
        )

        known_flawed_placeholder = (
            existing_acceptance_hash
            == known_flawed_acceptance_sha256

            and existing_acceptance_json.get(
                "decision",
                {},
            ).get(
                "next_notebook"
            )
            == (
                "notebooks/phase2/"
                "09_..."
            )
        )

        assert known_flawed_placeholder, (
            "Existing NB08 acceptance artifact "
            "differs from the corrected candidate "
            "and is not the known placeholder version. "
            "Refusing overwrite."
        )

        normalization_acceptance_path.write_text(
            candidate_acceptance_text,
            encoding="utf-8",
        )

        acceptance_write_action = (
            "CORRECTED_KNOWN_PLACEHOLDER"
        )

else:
    normalization_acceptance_path.write_text(
        candidate_acceptance_text,
        encoding="utf-8",
    )

    acceptance_write_action = "CREATED"


acceptance_sha256 = (
    acceptance_file_sha256(
        normalization_acceptance_path
    )
)

acceptance_size_bytes = (
    normalization_acceptance_path
    .stat()
    .st_size
)


# ---------------------------------------------------------
# Independent reload
# ---------------------------------------------------------

acceptance_disk = json.loads(
    normalization_acceptance_path.read_text(
        encoding="utf-8"
    )
)


assert (
    acceptance_disk[
        "acceptance_status"
    ]
    == "PASS"
)

assert (
    acceptance_disk[
        "decision"
    ][
        "status"
    ]
    == "PASS"
)

assert (
    acceptance_disk[
        "decision"
    ][
        "next_notebook_allowed"
    ]
    is True
)

assert (
    acceptance_disk[
        "decision"
    ][
        "scope_boundary"
    ]
    == "NB08_COMPLETE_NB09_NOT_STARTED"
)

assert (
    "next_notebook"
    not in acceptance_disk[
        "decision"
    ]
)

assert all(
    value == "PASS"
    for value in acceptance_disk[
        "quality_gates"
    ].values()
)

assert (
    acceptance_disk[
        "artifacts"
    ][
        "normalized_dataset"
    ][
        "sha256"
    ]
    == acceptance_file_sha256(
        normalized_dataset_path
    )
)

assert (
    acceptance_disk[
        "artifacts"
    ][
        "normalization_sidecar"
    ][
        "sha256"
    ]
    == acceptance_file_sha256(
        normalization_sidecar_path
    )
)

assert (
    acceptance_disk[
        "artifacts"
    ][
        "technical_report"
    ][
        "sha256"
    ]
    == acceptance_file_sha256(
        normalization_report_path
    )
)


print("NB08 FINAL ACCEPTANCE")
print("=" * 90)

print(
    "Path:",
    normalization_acceptance_path.relative_to(
        repo_root
    ),
)

print(
    "Action:",
    acceptance_write_action,
)

print(
    "Size bytes:",
    acceptance_size_bytes,
)

print(
    "SHA-256:",
    acceptance_sha256,
)

print(
    "Acceptance status:",
    acceptance_disk[
        "acceptance_status"
    ],
)

print(
    "Quality gates:",
    len(
        acceptance_disk[
            "quality_gates"
        ]
    ),
)

print(
    "Failed quality gates:",
    0,
)

print(
    "Next notebook allowed:",
    acceptance_disk[
        "decision"
    ][
        "next_notebook_allowed"
    ],
)

print(
    "Scope boundary:",
    acceptance_disk[
        "decision"
    ][
        "scope_boundary"
    ],
)

print(
    "\nNB08 PERSIAN NORMALIZATION "
    "FINAL ACCEPTANCE: PASS"
)

NB08 FINAL ACCEPTANCE
Path: reports/phase2/persian_normalization_acceptance_v1.json
Action: REUSED_IDENTICAL
Size bytes: 3637
SHA-256: e4de079bda42e34c1fd05ecbb3a7d2a4101fe0ccfe4e2a3560dd60fe354d57bc
Acceptance status: PASS
Quality gates: 20
Failed quality gates: 0
Next notebook allowed: True
Scope boundary: NB08_COMPLETE_NB09_NOT_STARTED

NB08 PERSIAN NORMALIZATION FINAL ACCEPTANCE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>جمع‌بندی و بستن Scope نوت‌بوک 08</h3>

<p>
Notebook 08 عملیات Persian Normalization را روی کل Common Document Corpus مرحله دوم اجرا و اعتبارسنجی کرده است.
</p>

<p>
خروجی نهایی شامل Dataset نرمال‌شده، Sidecar کامل Raw/Normalized، Transformation Log، گزارش فنی و Acceptance رسمی است.
</p>

<h4>نتیجه نهایی</h4>

<ul>
<li>19,456 Document پردازش شد.</li>
<li>263 Document واقعاً تغییر کرد.</li>
<li>Schema، Hash، Lineage و Source-specific Metadata بدون Failure حفظ شدند.</li>
<li>تمام پاک‌سازی‌های الزامی Unicode به صفر Residual رسیدند.</li>
<li>ی و ک عربی به دلیل نبود Rule ایمن Global Conversion نشدند و Preserve/Defer شدند.</li>
<li>Raw Text برای تمام رکوردها در Sidecar حفظ شده است.</li>
<li>هیچ Deduplication، Chunking، Embedding یا Indexing در NB08 انجام نشده است.</li>
<li>Scope نهایی در NB08 بسته می‌شود و NB09 در این Notebook شروع نشده است.</li>
</ul>

</div>

In [23]:
# Code Cell 23

import hashlib
import json


def final_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ---------------------------------------------------------
# Reload final persisted contracts
# ---------------------------------------------------------

final_report = json.loads(
    normalization_report_path.read_text(
        encoding="utf-8"
    )
)

final_acceptance = json.loads(
    normalization_acceptance_path.read_text(
        encoding="utf-8"
    )
)


# ---------------------------------------------------------
# Final artifact existence
# ---------------------------------------------------------

final_artifacts = {
    "normalized_dataset": normalized_dataset_path,
    "normalization_sidecar": normalization_sidecar_path,
    "technical_report": normalization_report_path,
    "acceptance_report": normalization_acceptance_path,
}

missing_final_artifacts = [
    name
    for name, path in final_artifacts.items()
    if not path.is_file()
]

assert not missing_final_artifacts, (
    "Missing final NB08 artifacts: "
    f"{missing_final_artifacts}"
)


# ---------------------------------------------------------
# Final contract gates
# ---------------------------------------------------------

assert (
    final_report[
        "validation"
    ][
        "status"
    ]
    == "PASS"
)

assert (
    final_acceptance[
        "acceptance_status"
    ]
    == "PASS"
)

assert (
    final_acceptance[
        "decision"
    ][
        "status"
    ]
    == "PASS"
)

assert (
    final_acceptance[
        "decision"
    ][
        "scope_boundary"
    ]
    == "NB08_COMPLETE_NB09_NOT_STARTED"
)

assert (
    final_acceptance[
        "decision"
    ][
        "next_notebook_allowed"
    ]
    is True
)

assert (
    final_acceptance[
        "population"
    ][
        "records"
    ]
    == 19456
)

assert (
    final_acceptance[
        "population"
    ][
        "changed_documents"
    ]
    == 263
)

assert (
    final_acceptance[
        "population"
    ][
        "unchanged_documents"
    ]
    == 19193
)

assert all(
    value == "PASS"
    for value in final_acceptance[
        "quality_gates"
    ].values()
)


# ---------------------------------------------------------
# Final hash reconciliation
# ---------------------------------------------------------

assert (
    final_acceptance[
        "artifacts"
    ][
        "normalized_dataset"
    ][
        "sha256"
    ]
    == final_sha256(
        normalized_dataset_path
    )
)

assert (
    final_acceptance[
        "artifacts"
    ][
        "normalization_sidecar"
    ][
        "sha256"
    ]
    == final_sha256(
        normalization_sidecar_path
    )
)

assert (
    final_acceptance[
        "artifacts"
    ][
        "technical_report"
    ][
        "sha256"
    ]
    == final_sha256(
        normalization_report_path
    )
)


# ---------------------------------------------------------
# Final scope protection
# ---------------------------------------------------------

assert (
    final_acceptance[
        "processing_boundaries"
    ][
        "deduplication"
    ]
    == "NOT_PERFORMED_IN_NB08"
)

assert (
    final_acceptance[
        "processing_boundaries"
    ][
        "chunking"
    ]
    == "NOT_PERFORMED_IN_NB08"
)

assert (
    final_acceptance[
        "processing_boundaries"
    ][
        "embedding"
    ]
    == "NOT_PERFORMED_IN_NB08"
)

assert (
    final_acceptance[
        "processing_boundaries"
    ][
        "indexing"
    ]
    == "NOT_PERFORMED_IN_NB08"
)


# ---------------------------------------------------------
# Final summary
# ---------------------------------------------------------

print("NB08 FINAL NOTEBOOK SUMMARY")
print("=" * 90)

print(
    "Records:",
    final_acceptance[
        "population"
    ][
        "records"
    ],
)

print(
    "Changed documents:",
    final_acceptance[
        "population"
    ][
        "changed_documents"
    ],
)

print(
    "Quality gates:",
    len(
        final_acceptance[
            "quality_gates"
        ]
    ),
)

print(
    "Acceptance:",
    final_acceptance[
        "acceptance_status"
    ],
)

print(
    "Policy:",
    (
        final_acceptance[
            "policy"
        ][
            "policy_version"
        ]
        + " / "
        + final_acceptance[
            "policy"
        ][
            "policy_revision"
        ]
    ),
)

print("\nFINAL ARTIFACTS")

for name, path in final_artifacts.items():
    print(
        f"  {name}: "
        f"{path.relative_to(repo_root)}"
    )
    print(
        f"    SHA-256: "
        f"{final_sha256(path)}"
    )

print(
    "\nScope boundary:",
    final_acceptance[
        "decision"
    ][
        "scope_boundary"
    ],
)

print(
    "\nNB08 NOTEBOOK SCOPE CLOSURE: PASS"
)

NB08 FINAL NOTEBOOK SUMMARY
Records: 19456
Changed documents: 263
Quality gates: 20
Acceptance: PASS
Policy: persian_normalization_v1 / contextual_controls_r2

FINAL ARTIFACTS
  normalized_dataset: data/phase2/processed/documents_normalized_v1.jsonl
    SHA-256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
  normalization_sidecar: data/phase2/processed/persian_normalization_records_v1.jsonl
    SHA-256: ea73c8e9b1547c11c58a0f5b845a0ca8e5932a6d72392ff81feb44f9404e6ef7
  technical_report: reports/phase2/persian_normalization_report_v1.json
    SHA-256: bd3ffd58bb97dd460970aa9bd9043bf85690214d4614557dc384de611dd761ca
  acceptance_report: reports/phase2/persian_normalization_acceptance_v1.json
    SHA-256: e4de079bda42e34c1fd05ecbb3a7d2a4101fe0ccfe4e2a3560dd60fe354d57bc

Scope boundary: NB08_COMPLETE_NB09_NOT_STARTED

NB08 NOTEBOOK SCOPE CLOSURE: PASS
